# Financial Data Agent Workshop — Fraud & AML

**Build a memory-aware AML data agent — a Meridian Bank analyst — on Oracle AI Database 26ai in 90 minutes.**

Oracle is already provisioned (`app/scripts/bootstrap.py` → `seed.py` → `setup_advanced.py` ran on first launch). This notebook builds the five pieces of the harness: memory, retrieval, a vector-indexed toolbox and skillbox, context engineering, and the agent loop.

| Already in Oracle | You build |
|---|---|
| `AGENT` user, vector pool, ONNX embedder + reranker | `OracleONNXEmbedder` + the OAMP wiring |
| `FINANCE` (Meridian Bank: branches, customers, accounts, cards, merchants, transactions, loans, SAR reports) | the scanner that turns catalog views into memories |
| Oracle Text index on the OAMP body | semantic, reranked, and hybrid retrieval |
| `toolbox` + `skillbox`, seeded from [`oracle/skills`](https://github.com/oracle/skills) | `@register`, the safe SQL tool, skill search |
| A running Flask + React app against the same Oracle | `agent_turn` — the loop |

![Meridian Bank AML agent — high-level architecture](images/architecture-excalidraw.png)

Answers: [`notebook_complete.ipynb`](notebook_complete.ipynb) · guides: [`docs/`](docs/) · Oracle provisioning: `app/scripts/`.


<!-- workshop:self-check:begin -->

## 0.1 Run this first — kernel check

Confirms the notebook kernel has the workshop dependencies and prints the URL of the running app. If it fails with `ModuleNotFoundError: No module named 'oracledb'`, the kernel is wrong: Command Palette → **Notebook: Select Notebook Kernel** → **Python 3.11** (`/usr/local/bin/python`).

In [ ]:
# §0.1 — Run me first: kernel check + where the running app lives.
import importlib.util, os, platform, sys

_missing = [m for m in ("numpy", "oracledb", "openai", "oracleagentmemory")
            if importlib.util.find_spec(m) is None]
if _missing:
    raise RuntimeError(
        f"This kernel is missing: {', '.join(_missing)}.\n"
        "Pick the workshop interpreter: Command Palette -> 'Notebook: Select Notebook Kernel' ->\n"
        "Python 3.11 at /usr/local/bin/python (see images/select_kernel.png).\n"
        "Everything is pre-installed for that interpreter — do not pip install."
    )

print(f"✅ kernel: Python {platform.python_version()} · {sys.executable}")
print(f"✅ working directory: {os.getcwd()}")

_codespace = os.environ.get("CODESPACE_NAME", "").strip()
_domain = os.environ.get("GITHUB_CODESPACES_PORT_FORWARDING_DOMAIN", "").strip()
print("\nRunning app — same Oracle, same memory store this notebook writes to:")
if _codespace and _domain:
    print(f"  forwarded: https://{_codespace}-3000.{_domain}   "
          f"(API health: https://{_codespace}-8000.{_domain}/api/health)")
print("  localhost: http://localhost:3000   (API health: http://localhost:8000/api/health)")
print("  If the preview did not open, use the PORTS tab.")

<!-- workshop:self-check:end -->

## What is an "agent harness"?

A model alone is tokens in, tokens out: no memory, no tools, no state between calls.

```
Agent  =  Model  +  Harness
```

Everything else — memory, retrieval, tool dispatch, identity, budgets — is the **harness**, and most "agent quality" complaints are harness problems. This notebook builds one for an AML data agent.


### How we build it — five blocks

Each block produces what the next one consumes.

| Block | What it adds | Needs |
|---|---|---|
| 1 | Setup, the `AGENT` connection, an OpenAI-compatible chat helper | — |
| 2 | Long-term memory (OAMP) + a scanner that mines Oracle's catalog into memories | Block 1 |
| 3 | Retrieval — cosine, cross-encoder rerank, hybrid RRF | Block 2 |
| 4 | Tools (`@register`) + skills (markdown playbooks) | Block 3 |
| 5 | Context engineering + the agent loop | Blocks 1–4 |

The guides in `docs/` extend the same harness with DBFS, Oracle MLE, duality views, identity-aware authorization, and tool-output offload.


## At a glance — the 9 TODOs

Oracle, the models, and the seed are done, so every TODO is harness code. Each has a **hard-stop assert** below it: break a TODO and the next cell raises instead of letting broken state run on.

| Block | Topic | TODO |
|---|---|---|
| 1 | Setup, Oracle connectivity, an OpenAI-compatible chat helper | **TODO 1** — ask the bare model a question |
| 2 | Long-term memory with OAMP | **TODO 2** — `OracleONNXEmbedder.embed` · **TODO 3** — `_scan_tables` |
| 3 | Retrieval | **TODO 4** — `retrieve_knowledge` · **TODO 5** — `hybrid_rrf_search_memories` |
| 4 | Toolbox + skillbox | **TODO 6** — `retrieve_tools` · **TODO 7** — `tool_run_sql` · **TODO 8** — `tool_list_skills` |
| 5 | Context engineering + the agent loop | **TODO 9** — `agent_turn` |

`Run All` halts by design. Checklist: [`docs/TODO-checklist.md`](docs/TODO-checklist.md).

The same harness is already running as the Flask + React app at **http://localhost:3000** — same Oracle, same memory store, same `toolbox` / `skillbox`: [`app/README.md`](app/README.md).


# Part 1 — Setup

> 📖 Guide: [`docs/part-1-setup.md`](docs/part-1-setup.md) · 🔧 **TODO 1** — ask the bare model a question

First launch provisioned Oracle: the `AGENT` user with its grants, `vector_memory_size = 512M`, the in-database ONNX embedder (`ALL_MINILM_L12_V2`) and reranker (`RERANKER_ONNX`), the seeded `FINANCE` schema (Meridian Bank, ~40,000 rows across 15 tables), the `account_dv` / `customer_dv` duality views, the Oracle Text index on the memory body, and the skillbox.

No DDL below — these cells open a connection and wire up the chat client.


## 1.1 Imports + credentials

Only what the notebook needs. `LLM_PROVIDER` picks the chat provider (`openai` or `oci`); both speak the OpenAI wire protocol through the `openai` SDK.


In [ ]:
import os, sys, time, json, re, hashlib, uuid, getpass, warnings, concurrent.futures
from pathlib import Path
import numpy as np
import oracledb
from dataclasses import dataclass
from openai import OpenAI

warnings.filterwarnings("ignore", category=UserWarning, module="oracleagentmemory")

# The shared helper keeps the notebook path and the app path behavior identical.
_repo_root = next((candidate for candidate in [Path.cwd(), *Path.cwd().parents]
                   if (candidate / "oci_key_rotation.py").exists()), None)
if _repo_root is None:
    raise RuntimeError("Run this notebook from the workshop repository")
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
from oci_key_rotation import KeyRotator, call_with_failover, load_oci_keys, make_rotating_oamp_llm

LLM_PROVIDER = os.environ.setdefault("LLM_PROVIDER", "oci").lower()
assert LLM_PROVIDER in ("openai", "oci"), "LLM_PROVIDER must be 'openai' or 'oci'"

if LLM_PROVIDER == "oci":
    if not load_oci_keys():
        os.environ["OCI_GENAI_API_KEY"] = getpass.getpass("OCI GenAI API key: ")
    os.environ.setdefault(
        "OCI_GENAI_ENDPOINT",
        "https://inference.generativeai.us-phoenix-1.oci.oraclecloud.com/openai/v1",
    )
    os.environ.setdefault("LLM_MODEL", "xai.grok-4.3")
    _oci_keys = load_oci_keys()
    OCI_ROTATOR = KeyRotator(_oci_keys)
    print(f"OCI key rotation enabled: {len(OCI_ROTATOR)} key(s), starting at key #{OCI_ROTATOR.current_index() + 1}")
else:
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
    os.environ.setdefault("LLM_MODEL", "gpt-5.5")
    OCI_ROTATOR = None

OCI_ENDPOINT = os.environ.get("OCI_GENAI_ENDPOINT", "").rstrip("/")
if LLM_PROVIDER == "oci" and not OCI_ENDPOINT.endswith("/openai/v1"):
    OCI_ENDPOINT = f"{OCI_ENDPOINT}/openai/v1"
    os.environ["OCI_GENAI_ENDPOINT"] = OCI_ENDPOINT
OCI_COMPARTMENT_ID = os.environ.get("OCI_COMPARTMENT_ID", "").strip()
print(f"LLM: {LLM_PROVIDER}/{os.environ['LLM_MODEL']}")


## 1.2 Connect to the `AGENT` user

`oracledb.connect` behind a small retry wrapper. Three database users matter:

| Oracle user | Owns | Used by |
|---|---|---|
| `SYS` | the whole database | one-time bootstrap |
| `AGENT` | the harness's state — OAMP memory, `toolbox` / `skillbox`, the DBFS scratchpad | every cell below |
| `FINANCE` | the business data — branches, customers, accounts, transactions, `sar_reports` | the agent, read-only, via `tool_run_sql` (Part 6) |

That split is the trust boundary: `AGENT` reads `FINANCE` but owns nothing in it, so no prompt can drop the bank's tables. ("User" is overloaded here — OAMP also tags memories with a `user_id` (`USER_ID = "enterprise-operator"` in Part 2); that is a scoping label, not a database login.)


In [ ]:
# Same defaults the Codespace exports (devcontainer.json `remoteEnv`), overridable
# so the notebook also runs against a local Oracle on another DSN.
SYS_DSN    = os.environ.get("ORACLE_DSN", "localhost:1521/FREEPDB1")
AGENT_USER = os.environ.get("ORACLE_AGENT_USER", "AGENT")
AGENT_PASS = os.environ.get("ORACLE_AGENT_PASS", "AgentPwd_2025")
DEMO_USER  = os.environ.get("ORACLE_DEMO_USER", "FINANCE")

def connect(user, password, dsn, mode=None, retries=5):
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            kwargs = dict(user=user, password=password, dsn=dsn)
            if mode is not None:
                kwargs["mode"] = mode
            conn = oracledb.connect(**kwargs)
            with conn.cursor() as cur:
                cur.execute("SELECT banner FROM v$version WHERE rownum = 1")
                print(f"connected as {user}@{dsn} — {cur.fetchone()[0]}")
            return conn
        except Exception as e:
            last_err = e
            time.sleep(3)
    raise RuntimeError(f"could not connect to {dsn}: {last_err}")


agent_conn = connect(AGENT_USER, AGENT_PASS, SYS_DSN)

<!-- workshop:preflight:begin -->

## 1.3 Preflight — is this Codespace ready?

Twelve checks in one table: the `FINANCE` seed, the ONNX embedder and reranker, the Oracle Text index, `toolbox` / `skillbox`, the API keys. Every ❌ prints the command that fixes it; the two checks the rest of the notebook cannot survive raise instead of warning.

In [ ]:
# §1.3 — Preflight. Reads the same catalogs the agent will scan, and the same env the
# harness reads. Nothing is written; run it as often as you like.
from importlib.metadata import version as _pkg_version


def _scalar(sql, default=None, **binds):
    """Single-value query; returns `default` if the object does not exist yet."""
    try:
        with agent_conn.cursor() as cur:
            cur.execute(sql, **binds)
            return cur.fetchone()[0]
    except oracledb.DatabaseError:
        return default


def _catalog(query, **binds):
    """Self-contained row fetch — the preflight runs before Part 12 defines `_rows`."""
    with agent_conn.cursor() as cur:
        cur.execute(query, **binds)
        return cur.fetchall()


def _check(label, ok, detail="", fix=""):
    print(f"  {'✅' if ok else '❌'} {label:<32} {detail}")
    if not ok and fix:
        print(f"     ↳ fix: {fix}")
    return ok


print("Environment")
print(f"  ✅ {'python':<32} {sys.version.split()[0]} at {sys.executable}")
for _pkg in ("oracledb", "oracleagentmemory", "openai", "numpy"):
    try:
        print(f"  ✅ {_pkg:<32} {_pkg_version(_pkg)}")
    except Exception:
        print(f"  ⚠️  {_pkg:<32} version unknown")

print("\nOracle — AGENT schema (the harness's own state)")
_mem_table = _scalar("SELECT COUNT(*) FROM user_tables WHERE table_name = 'EDA_ONNX_MEMORY'", 0)
_mem_rows = _scalar("SELECT COUNT(*) FROM eda_onnx_memory", 0) if _mem_table else 0
print(f"  ✅ {'OAMP memory table':<32} EDA_ONNX_MEMORY · {_mem_rows} memor{'y' if _mem_rows == 1 else 'ies'}")
_text_idx = _scalar("SELECT COUNT(*) FROM user_indexes WHERE index_name = 'EDA_MEMORY_TEXT_IDX'", 0)
print(f"  {'✅' if _text_idx else '⚠️ '} {'Oracle Text index':<32} "
      f"{'EDA_MEMORY_TEXT_IDX present' if _text_idx else 'absent — §3.3a creates it (keyword leg needs it)'}")
_toolbox = _scalar("SELECT COUNT(*) FROM toolbox", None)
_skillbox = _scalar("SELECT COUNT(*) FROM skillbox", None)
print(f"  {'✅' if _toolbox is not None else '⚠️ '} {'toolbox / skillbox':<32} "
      f"{_toolbox if _toolbox is not None else 'missing'} tools · "
      f"{_skillbox if _skillbox is not None else 'missing'} skills")

print("\nOracle — FINANCE (the bank's data; the agent only ever reads it)")
FINTECH_TABLES = ("BRANCHES", "CUSTOMERS", "ACCOUNTS", "CARDS", "MERCHANTS",
                  "TRANSACTIONS", "LOANS", "SAR_REPORTS", "SANCTIONS_SCREENINGS",
                  "BENEFICIAL_OWNERS", "WIRE_MESSAGES", "LOGIN_EVENTS",
                  "KYC_DOCUMENTS", "CASE_NOTES", "FX_RATES")
_present = {row[0] for row in _catalog(
    "SELECT table_name FROM all_tables WHERE owner = 'FINANCE'") if "$" not in row[0]}
_missing = [t for t in FINTECH_TABLES if t not in _present]
_txns = _scalar("SELECT COUNT(*) FROM finance.transactions", 0)
_flagged = _scalar("SELECT COUNT(*) FROM finance.transactions "
                   "WHERE status IN ('FLAGGED','BLOCKED') AND flag_reason IS NOT NULL", 0)
_sars = _scalar("SELECT COUNT(*) FROM finance.sar_reports", 0)
_seed_fix = "cd app && python scripts/bootstrap.py && python scripts/seed.py"
_check("FINANCE business tables", not _missing,
       f"{len(FINTECH_TABLES) - len(_missing)}/{len(FINTECH_TABLES)} present (Spatial index tables ignored)",
       fix=f"missing {', '.join(_missing)} — " + _seed_fix if _missing else "")
_alert_queue = _scalar(
    "SELECT COUNT(*) FROM (SELECT a.customer_id, t.flag_reason "
    "  FROM finance.transactions t JOIN finance.accounts a ON a.account_id = t.account_id "
    " WHERE t.status IN ('FLAGGED','BLOCKED') "
    "   AND t.txn_ts >= SYSTIMESTAMP - INTERVAL '30' DAY "
    " GROUP BY a.customer_id, t.flag_reason)", 0)
_check("transactions", _txns > 5000,
       f"{_txns} rows (Meridian Bank seed ≈ 23,600), {_flagged} flagged/blocked", fix=_seed_fix)
_check("AML alerts in the seed", _flagged > 0,
       f"{_alert_queue} alerts in the 30-day window (Part 12's queue)", fix=_seed_fix)
_check("SAR reports", _sars > 0, f"{_sars} reports (compliance-only table)", fix=_seed_fix)
_ops = {t: _scalar(f"SELECT COUNT(*) FROM finance.{t}", 0) for t in
        ("sanctions_screenings", "beneficial_owners", "wire_messages", "login_events",
         "kyc_documents", "case_notes", "fx_rates")}
_check("AML operational tables", all(v > 0 for v in _ops.values()),
       " · ".join(f"{k.replace('_', ' ')} {v}" for k, v in _ops.items()), fix=_seed_fix)

print("\nIn-database models")
_embedder = _scalar("SELECT COUNT(*) FROM user_mining_models WHERE model_name = 'ALL_MINILM_L12_V2'", 0)
_reranker = _scalar("SELECT COUNT(*) FROM user_mining_models WHERE model_name = 'RERANKER_ONNX'", 0)
_check("ONNX embedder", bool(_embedder), "ALL_MINILM_L12_V2 (384-dim, in-database)",
       fix="cd app && python scripts/bootstrap.py")
print(f"  {'✅' if _reranker else '⚠️ '} {'ONNX reranker':<32} "
      f"{'RERANKER_ONNX present' if _reranker else 'absent — retrieval falls back to cosine ordering (fine)'}")

print("\nChat LLM")
_keys = len(OCI_ROTATOR) if LLM_PROVIDER == "oci" and OCI_ROTATOR else 0
if LLM_PROVIDER == "openai":
    _keys = 1 if os.environ.get("OPENAI_API_KEY") else 0
print(f"  {'✅' if _keys else '❌'} {'credentials':<32} provider {LLM_PROVIDER} · model {os.environ['LLM_MODEL']} · {_keys} key(s)")
if LLM_PROVIDER == "oci":
    print(f"     endpoint: {OCI_ENDPOINT}")
if not _keys:
    print("     ↳ fix: add OCI_GENAI_API_KEY as a Codespaces secret (then restart), or")
    print("            echo 'OCI_GENAI_API_KEY=...' >> app/.env  — TODO 1 is the first live call.")

# The two hard requirements: without these, Blocks 2–12 cannot run.
if _missing or not _txns:
    raise RuntimeError("FINANCE is not seeded. Fix: " + _seed_fix)
if not _embedder:
    raise RuntimeError("The in-database embedder is missing. Fix: cd app && python scripts/bootstrap.py")

print("\n✅ Preflight passed — every block below has what it needs.")

<!-- workshop:preflight:end -->

## 1.4 Chat LLM client

The standard `openai` SDK, pointed at OpenAI or at OCI GenAI's OpenAI-compatible endpoint (`LLM_PROVIDER`).


In [ ]:
LLM_MODEL = os.environ["LLM_MODEL"]


def _make_oci_client(key):
    options = {"base_url": OCI_ENDPOINT, "api_key": key}
    if OCI_COMPARTMENT_ID:
        options["project"] = OCI_COMPARTMENT_ID
    return OpenAI(**options)


if LLM_PROVIDER == "oci":
    llm = _make_oci_client(OCI_ROTATOR.current())
else:
    llm = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


def chat(messages, tools=None, model=LLM_MODEL, max_retries=3):
    kwargs = {"model": model, "messages": messages}
    if tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = "auto"

    if LLM_PROVIDER == "oci":
        return call_with_failover(
            _make_oci_client,
            lambda client: client.chat.completions.create(**kwargs),
            OCI_ROTATOR,
            base_delay=2.0,
            on_event=print,
        )

    delay = 2.0
    for attempt in range(max_retries + 1):
        try:
            return llm.chat.completions.create(**kwargs)
        except Exception as e:
            status = getattr(e, "status_code", None) or 0
            if status == 429 and attempt < max_retries:
                time.sleep(delay)
                delay *= 2
            else:
                raise


# TODO 1: talk to the bare model. Set QUESTION, then run the cell. No memory,
# no retrieval, no tools: just the reasoning core. By Part 7 the harness wraps
# this same call with everything you build in between.
# 📖 docs/part-1-setup.md

QUESTION = ""

_bare_answer = None
if QUESTION.strip():
    _bare_resp = chat([{"role": "user", "content": QUESTION}])
    _bare_answer = _bare_resp.choices[0].message.content
    print("Q:", QUESTION)
    print("A:", _bare_answer)
else:
    print("Set QUESTION above, then run this cell again.")

# Hard-stop checkpoint: TODO 1. The bare model must answer before the harness wraps it.
assert QUESTION.strip(), "❌ TODO 1: set QUESTION to a real question first."
assert _bare_answer, "❌ TODO 1: the model returned no answer. Check your key and LLM_MODEL."
print("✅ TODO 1 passed: the bare model answered with no memory, retrieval, or tools.")


# Part 2 — Long-Term Memory with OAMP

> 📖 Guide: [`docs/part-2-oamp-memory.md`](docs/part-2-oamp-memory.md) · 🔧 **TODOs 2–3** — `OracleONNXEmbedder.embed`, `_scan_tables`

Block 2 gives the harness somewhere durable to keep what it learns: the [Oracle AI Agent Memory Package](https://www.oracle.com/database/ai-agent-memory/) (OAMP).

| OAMP primitive | What it stores |
|---|---|
| `memory` | durable facts — scanned schema entries, corrections, tool outputs |
| `thread` | one conversation: messages + a context card |
| `context_card` | a compact, query-relevant block of memories and recent turns |

OAMP owns the DDL; `memory_store_id="EDA_ONNX"` names the tables (`EDA_ONNX_MEMORY`, `EDA_ONNX_THREAD`, …).


## 2.1 In-DB ONNX embedder for OAMP

An **embedder** turns text into a fixed-length vector, so similar *meanings* land near each other regardless of wording. We use `ALL_MINILM_L12_V2` (384 dimensions), loaded **into the database** by `bootstrap.py` via `DBMS_VECTOR.LOAD_ONNX_MODEL` and called as SQL: `VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :t AS DATA)`.

OAMP embeds on every write and every search, so keeping the model in-database means no embedding API, no network hop, and the same trust boundary as the data. The wrapper below adapts that SQL function to OAMP's `IEmbedder` interface. **TODO 2:** implement `OracleONNXEmbedder.embed`.


In [ ]:
from oracleagentmemory.core import OracleAgentMemory, MemoryExtractionConfig
from oracleagentmemory.core.llms import Llm
from oracleagentmemory.apis.embedders.embedder import IEmbedder

ONNX_EMBED_MODEL = "ALL_MINILM_L12_V2"
ONNX_EMBED_DIM   = 384


class OracleONNXEmbedder(IEmbedder):
    def __init__(self, conn, model_name=ONNX_EMBED_MODEL, dim=ONNX_EMBED_DIM):
        self._conn = conn
        self._model = model_name
        self._dim = dim

    def embed(self, texts, *, is_query=False):
        # TODO 2: embed every text INSIDE the database and return an
        # (len(texts), ONNX_EMBED_DIM) float32 array. One SELECT per text.
        # 📖 docs/part-2-oamp-memory.md
        raise NotImplementedError("TODO 2: OracleONNXEmbedder.embed")

    async def embed_async(self, texts, *, is_query=False):
        return self.embed(texts, is_query=is_query)

# Hard-stop checkpoint: TODO 2. OAMP embeds on every write and every search, so
# this must pass before the memory client below is built.
_embed_probe = OracleONNXEmbedder(agent_conn).embed(["transaction amount in cents"])
assert _embed_probe.shape == (1, ONNX_EMBED_DIM), "❌ TODO 2: expected one vector per text."
assert float((_embed_probe ** 2).sum()) > 0.0, "❌ TODO 2: embedder returned an all-zero vector."
print(f"✅ TODO 2 passed: in-DB embedder returned shape {_embed_probe.shape}.")


## 2.2 Why `Llm("openai/…")` even when `LLM_PROVIDER=oci`

OAMP runs on LiteLLM, where `Llm("openai/<model>", api_base=…)` means *use the OpenAI-compatible client, pointed here* — not *call OpenAI*. OCI GenAI speaks that protocol, so `api_base` picks the destination: the `openai/` prefix is the client, not the server.


In [ ]:
if LLM_PROVIDER == "oci":
    extraction_llm = make_rotating_oamp_llm(LLM_MODEL, OCI_ENDPOINT, OCI_ROTATOR)
else:
    extraction_llm = Llm(LLM_MODEL)                          # bare = real OpenAI


memory_client = OracleAgentMemory(
    connection=agent_conn,
    embedder=OracleONNXEmbedder(agent_conn),
    llm=extraction_llm,
    memory_extraction_config=MemoryExtractionConfig(extract_memories=True),
    schema_policy="create_if_necessary",
    memory_store_id="EDA_ONNX",
)


USER_ID  = "enterprise-operator"
AGENT_ID = "enterprise-data-agent"
for register_fn, eid, info in [
    (memory_client.add_user,  USER_ID,  "Operator querying the enterprise database in natural language."),
    (memory_client.add_agent, AGENT_ID, "Data agent grounded in scanned schema metadata."),
]:
    try:
        register_fn(eid, info)
    except ValueError as e:
        if "already exists" not in str(e):
            raise

print(f"OAMP wired (in-DB ONNX embedder + {LLM_PROVIDER} extraction LLM); user/agent registered.")

## Concept check — what OAMP replaces

`oracleagentmemory` is the whole memory layer — in-database vectors, threads, context cards, user/agent scoping, extraction, provenance — as tables you can query with SQL.

Below: the same fact kept in a Python dict and matched by keyword, then stored through OAMP and found by *meaning* on a rephrased question.

<!-- workshop:concept-demo:oamp -->


In [ ]:
# ============================================================================
# WITHOUT OAMP vs WITH OAMP — the memory layer you would otherwise build yourself
# ============================================================================
q = "monetary denomination scale"   # deliberately shares NO words with the facts below

# ---- WITHOUT: an in-kernel dict, matched by keyword overlap ----------------
naive_memory = [
    {"subject": "FINANCE.TRANSACTIONS.amount_cents",
     "body": "Transaction amounts are stored in USD cents, never dollars."},
    {"subject": "FINANCE.CUSTOMERS.risk_rating",
     "body": "Customer risk rating is a 1-100 score; higher means riskier."},
]

def naive_search(query):
    q_words = set(query.lower().split())
    ranked = []
    for row in naive_memory:
        overlap = len(q_words & set((row["subject"] + " " + row["body"]).lower().split()))
        if overlap:
            ranked.append((overlap, row["subject"]))
    return [s for _, s in sorted(ranked, reverse=True)]

print("WITHOUT OAMP — a Python dict, matched by keyword overlap:")
print("   query:", repr(q))
print("   hits :", naive_search(q) or "[]  <- zero token overlap, so it finds nothing")
print("   -> no meaning, no persistence, no user/agent scoping, no provenance.")

# ---- WITH: OAMP, embedded and searchable by meaning ------------------------
memory_client.add_memory(
    "Table FINANCE.TRANSACTIONS column AMOUNT_CENTS: amounts are stored in USD CENTS, never dollars.",
    user_id=USER_ID, agent_id=AGENT_ID,
    metadata={"kind": "column", "subject": "FINANCE.TRANSACTIONS.AMOUNT_CENTS"},
)
hits = memory_client.search(q, user_id=USER_ID, agent_id=AGENT_ID,
                            record_types=["memory"], max_results=3)
print("\nWITH OAMP — the same query, searched by meaning:")
for h in hits:
    d = getattr(h, "distance", None)
    subject = (h.metadata or {}).get("subject", "?")
    print(f"   distance={float(d):.4f}  {subject}" if d is not None else f"   {subject}")
print("   -> the fact is retrieved despite zero shared keywords, stored in the EDA_ONNX_* tables,")
print("      scoped to user/agent, and reusable from any later turn or process.")


## 2.3 `Fact` dataclass and scanner helpers

A **scan** reads Oracle's catalog views, turns each entry into a sentence, embeds it, and stores it in OAMP. Without one, the agent invents tables and forgets that `amount_cents` is cents; with one, "which table records card transactions?" retrieves the fact about `FINANCE.TRANSACTIONS`, and every later query is grounded in catalog evidence.

| Source | Catalog views | Fact `kind` | Why we mine it |
|---|---|---|---|
| Structural | `ALL_TABLES`, `ALL_TAB_COMMENTS` | `table` | what shapes exist |
| Columns | `ALL_TAB_COLUMNS`, `ALL_COL_COMMENTS` | `column` | what each field means |
| Relational | `ALL_CONSTRAINTS`, `ALL_CONS_COLUMNS` | `relationship` | join paths |
| Workload | `V$SQL` | `query_pattern` | how the database is actually used |

Each helper takes `(conn, owner)` and returns `list[Fact]`; `write_facts` (later) turns each Fact into an embedded OAMP memory.


In [ ]:
@dataclass
class Fact:
    kind: str        # 'table' | 'column' | 'relationship' | 'query_pattern'
    subject: str     # e.g. 'FINANCE.TRANSACTIONS'
    body: str        # natural-language sentence the embedder will read
    metadata: dict



## 2.4 Implement `_scan_tables`

Mine `ALL_TABLES` + `ALL_TAB_COMMENTS` into one `Fact(kind="table")` per table. Build the body conditionally — skip the parts where the column is `None`.

> 📖 See: [Part 2 guide → TODO 3](docs/part-2-oamp-memory.md#todo-3-implement-_scan_tables)


In [ ]:
# TODO 3 — implement _scan_tables(conn, owner) -> list[Fact]
# Query: SELECT t.table_name, tc.comments, t.num_rows, t.last_analyzed
#          FROM all_tables t LEFT JOIN all_tab_comments tc USING(owner, table_name)
#         WHERE t.owner = :owner ORDER BY t.table_name
# For each row, build a body string and append a Fact(kind="table", subject=f"{owner}.{table}", ...).
# 📖 docs/part-2-oamp-memory.md

def _scan_tables(conn, owner: str) -> list[Fact]:
    # YOUR CODE HERE
    raise NotImplementedError("TODO 3: _scan_tables")


In [ ]:
def _scan_columns(conn, owner):
    sql = (
        "SELECT c.table_name, c.column_name, c.data_type, c.data_length, "
        "       c.nullable, cc.comments "
        "  FROM all_tab_columns c "
        "  LEFT JOIN all_col_comments cc "
        "    ON cc.owner = c.owner AND cc.table_name = c.table_name "
        "   AND cc.column_name = c.column_name "
        " WHERE c.owner = :owner "
        " ORDER BY c.table_name, c.column_id"
    )
    facts = []
    with conn.cursor() as cur:
        cur.execute(sql, owner=owner.upper())
        for table, col, dtype, dlen, nullable, comment in cur:
            dtype_str = dtype + (f"({dlen})" if dlen and dtype in ("VARCHAR2", "CHAR") else "")
            nullstr = "nullable" if nullable == "Y" else "NOT NULL"
            body = f"Column {owner}.{table}.{col} of type {dtype_str} ({nullstr})."
            if comment:
                body += f" Meaning: {comment}"
            facts.append(Fact("column", f"{owner}.{table}.{col}", body,
                              {"owner": owner, "table": table, "column": col,
                               "data_type": dtype, "nullable": nullable == "Y"}))
    return facts


In [ ]:
def _scan_relationships(conn, owner):
    sql = (
        "SELECT c.constraint_name, c.table_name, acc.column_name, "
        "       rc.table_name AS r_table, rcc.column_name AS r_column "
        "  FROM all_constraints c "
        "  JOIN all_cons_columns acc "
        "    ON acc.owner = c.owner AND acc.constraint_name = c.constraint_name "
        "  JOIN all_constraints rc "
        "    ON rc.owner = c.r_owner AND rc.constraint_name = c.r_constraint_name "
        "  JOIN all_cons_columns rcc "
        "    ON rcc.owner = rc.owner AND rcc.constraint_name = rc.constraint_name "
        "   AND rcc.position = acc.position "
        " WHERE c.owner = :owner AND c.constraint_type = 'R'"
    )
    facts = []
    with conn.cursor() as cur:
        cur.execute(sql, owner=owner.upper())
        for cname, tbl, col, r_tbl, r_col in cur:
            facts.append(Fact("relationship",
                              f"{owner}.{tbl}.{col}->{owner}.{r_tbl}.{r_col}",
                              f"Foreign key {cname}: {owner}.{tbl}.{col} references "
                              f"{owner}.{r_tbl}.{r_col}. Use this edge when joining the two tables.",
                              {"owner": owner, "child": tbl, "child_col": col,
                               "parent": r_tbl, "parent_col": r_col}))
    return facts

In [ ]:
def _scan_workload(conn, owner, limit=50):
    sql = (
        "SELECT sql_id, sql_fulltext, executions, rows_processed "
        "  FROM v$sql "
        " WHERE parsing_schema_name = :owner "
        "   AND command_type IN (3, 6, 7, 189) "
        "   AND rownum <= :lim "
        " ORDER BY executions DESC NULLS LAST"
    )
    facts = []
    with conn.cursor() as cur:
        try:
            cur.execute(sql, owner=owner.upper(), lim=limit)
            for sql_id, text, execs, rows_proc in cur:
                if text is None:
                    continue
                stmt = (text.read() if hasattr(text, "read") else str(text)).strip()
                if len(stmt) > 2000:
                    stmt = stmt[:2000] + " /* ...truncated */"
                facts.append(Fact("query_pattern", f"v$sql:{sql_id}",
                                  f"A SQL statement observed in the workload for {owner} "
                                  f"(executed {execs or 0} times, {rows_proc or 0} rows): {stmt}",
                                  {"owner": owner, "sql_id": sql_id,
                                   "executions": execs, "rows_processed": rows_proc}))
        except oracledb.DatabaseError as e:
            print(f"  (workload scan skipped: {e})")
    return facts


print("scanner helpers ready: _scan_columns, _scan_relationships, _scan_workload")
print("(_scan_tables is your TODO 3 — defined in the next cell)")

In [ ]:
# Hard-stop checkpoint: TODO 3 must work before later parts run.
_facts = _scan_tables(agent_conn, DEMO_USER)
assert _facts, "❌ TODO 3: _scan_tables returned no facts. FINANCE should have 8 tables."
assert isinstance(_facts[0], Fact), "❌ TODO 3: must return Fact objects, not dicts/tuples."
assert _facts[0].kind == "table", "❌ TODO 3: Facts must have kind='table'."
print(f"✅ TODO 3 passed — _scan_tables found {len(_facts)} tables in {DEMO_USER}.")

## 2.5 `scan_schema` and `write_facts`

`scan_schema` orchestrates the four scanners. `write_facts` does the idempotent upsert into OAMP: one current row per `(kind, subject)`, deduped by `body_hash`.


In [ ]:
def scan_schema(conn, owner):
    facts = []
    facts += _scan_tables(conn, owner)
    facts += _scan_columns(conn, owner)
    facts += _scan_relationships(conn, owner)
    facts += _scan_workload(conn, owner)
    return facts


def _hash(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:32]


def write_facts(facts, scan_id=None):
    new = updated = skipped = 0
    scan_id = scan_id or str(uuid.uuid4())
    store = memory_client._store

    for f in facts:
        body_hash = _hash(f.body)
        existing = store.list(
            "memory",
            user_id=USER_ID, agent_id=AGENT_ID,
            metadata_filter={"kind": f.kind, "subject": f.subject},
            limit=1,
        )
        meta = {**f.metadata, "kind": f.kind, "subject": f.subject,
                "body_hash": body_hash, "scan_id": scan_id}

        if not existing:
            memory_client.add_memory(f.body, user_id=USER_ID, agent_id=AGENT_ID, metadata=meta)
            new += 1
        elif (existing[0].metadata or {}).get("body_hash") == body_hash:
            skipped += 1
        else:
            memory_client.delete_memory(existing[0].id)
            memory_client.add_memory(f.body, user_id=USER_ID, agent_id=AGENT_ID, metadata=meta)
            updated += 1
    return new, updated, skipped


def run_scan(conn, owner):
    facts = scan_schema(conn, owner)
    new, updated, skipped = write_facts(facts)
    return {"facts_total": len(facts), "new": new, "updated": updated, "skipped": skipped}


summary = run_scan(agent_conn, DEMO_USER)
print(json.dumps(summary, indent=2))

# Part 3 — Retrieval

> 📖 Guide: [`docs/part-3-retrieval.md`](docs/part-3-retrieval.md) · 🔧 **TODOs 4–5** — `retrieve_knowledge`, `hybrid_rrf_search_memories`

Stored memories are only useful if they can be found. Two retrieval paths, both running server-side:

| Layer | What it gives you | When to use |
|---|---|---|
| Vector + rerank | cosine similarity, then cross-encoder rescoring | "where do we store transaction amounts?" |
| Hybrid (vector + Oracle Text + RRF) | both, fused by Reciprocal Rank Fusion | "amount_cents in cents" — concept plus exact token |


## 3.1 `rerank()` — cross-encoder rescoring

Cosine scores the query and each document *independently*; a **cross-encoder** reads them together and is far more accurate per pair, but slower. So: pull `k*4` candidates by cosine from the HNSW index, then rerank only those.

The reranker is in the database too — `PREDICTION(RERANKER_ONNX USING :q AS DATA1, :doc AS DATA2)` over a `JSON_TABLE` of candidates, one round trip. Without a reranker loaded, the helper falls back to cosine order.


In [ ]:
RERANKER_MODEL = "RERANKER_ONNX"


def _reranker_loaded():
    with agent_conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM user_mining_models WHERE model_name = :m",
                    m=RERANKER_MODEL)
        return cur.fetchone()[0] > 0


def rerank(query, candidates, top_k=5, content_key="body"):
    if not candidates:
        return candidates
    if not _reranker_loaded():
        return candidates[:top_k]
    docs = [{"index": i, "content": str(c.get(content_key, ""))[:1500]}
            for i, c in enumerate(candidates)]
    sql = (
        f"SELECT t.idx, PREDICTION({RERANKER_MODEL} USING :q AS DATA1, t.content AS DATA2) "
        "  FROM JSON_TABLE(:docs, '$[*]' COLUMNS ("
        "         idx NUMBER PATH '$.index', content VARCHAR2(4000) PATH '$.content')) t "
        " ORDER BY 2 DESC FETCH FIRST :k ROWS ONLY"
    )
    try:
        with agent_conn.cursor() as cur:
            cur.execute(sql, q=query, docs=json.dumps(docs), k=top_k)
            ranked = list(cur)
    except oracledb.DatabaseError:
        return candidates[:top_k]
    out = []
    for idx, score in ranked:
        if idx is None or int(idx) >= len(candidates):
            continue
        item = dict(candidates[int(idx)])
        item["rerank_score"] = float(score) if score is not None else 0.0
        out.append(item)
    return out


print(f"rerank ready (reranker loaded: {_reranker_loaded()})")

## 3.2 Implement `retrieve_knowledge`

1. Oversample 4×: `memory_client.search(..., max_results=k*4)`.
2. Drop `kind == "tool_output"`; apply the `kinds` filter if one was given.
3. `rerank(query, candidates, top_k=k, content_key="body")`.

> 📖 See: [Part 3 guide → TODO 4](docs/part-3-retrieval.md#todo-4-implement-retrieve_knowledge)


In [ ]:
# TODO 4 — implement retrieve_knowledge(query, k=5, kinds=None) -> list[dict]
# Two stages:
#   1. memory_client.search(...) with max_results = k*4
#   2. Filter out kind == "tool_output". Apply `kinds` filter if provided.
#   3. Rerank via rerank(query, candidates, top_k=k, content_key="body")
# 📖 docs/part-3-retrieval.md

def retrieve_knowledge(query: str, k: int = 5,
                       kinds: list[str] | None = None) -> list[dict]:
    """Semantic search over the agent's long-term memory."""
    # YOUR CODE HERE
    raise NotImplementedError("TODO 4: retrieve_knowledge")


In [ ]:
# Hard-stop checkpoint: TODO 4
_hits = retrieve_knowledge("which table holds transaction amounts", k=3)
assert _hits, "❌ TODO 4: retrieve_knowledge returned no hits. Did you call rerank() at the end?"
assert all("body" in h for h in _hits), "❌ TODO 4: each hit must have a 'body' key."
print(f"✅ TODO 4 passed — retrieve_knowledge returns {len(_hits)} reranked hits.")
for h in _hits:
    print(f"  [{h['kind']:12s}] {h['subject'][:40]:40s}  {h['body'][:80]}")

## 3.3 Keyword search

The keyword leg of the hybrid: `CONTAINS` over the `CTXSYS.CONTEXT` index on the OAMP body, scored by `SCORE(1)`, skipping offloaded `tool_output` rows. Oracle Text wants an *expression*, not free text — `_text_query` joins the words, or "transaction amount in cents" becomes a phrase search that matches nothing. Cell 3.3a recreates the index if it is missing, and the helper retries on `DRG-10599`.


In [ ]:
# 3.3a — Oracle Text index on the memory body. Required by the keyword leg.
#
# OAMP indexes its own chunk table, but the memory rows this workshop writes land in
# eda_onnx_memory.content, and CONTAINS() refuses to run on a column with no CTXSYS.CONTEXT
# index at all: "DRG-10599: column is not indexed". Create it if it is missing (idempotent).
MEMORY_TABLE    = "eda_onnx_memory"
TEXT_INDEX_NAME = "eda_memory_text_idx"

def _ensure_text_index(verbose: bool = True) -> bool:
    """Create the Oracle Text index on MEMORY_TABLE(content) if it is absent."""
    with agent_conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM user_indexes WHERE index_name = :n",
                    n=TEXT_INDEX_NAME.upper())
        if cur.fetchone()[0]:
            if verbose:
                print(f"  {TEXT_INDEX_NAME} already present")
            return False
        # SYNC (ON COMMIT) keeps the index current while the agent writes memories. It needs the
        # job queue, so fall back to a plain index where that parameter is rejected — then refresh
        # explicitly with ctx_ddl.sync_index (the setup grants EXECUTE ON CTX_DDL to AGENT).
        try:
            cur.execute(f"CREATE INDEX {TEXT_INDEX_NAME} ON {MEMORY_TABLE}(content) "
                        f"INDEXTYPE IS CTXSYS.CONTEXT PARAMETERS ('SYNC (ON COMMIT)')")
            if verbose:
                print(f"  created {TEXT_INDEX_NAME} (SYNC ON COMMIT)")
        except oracledb.DatabaseError:
            cur.execute(f"CREATE INDEX {TEXT_INDEX_NAME} ON {MEMORY_TABLE}(content) "
                        f"INDEXTYPE IS CTXSYS.CONTEXT")
            if verbose:
                print(f"  created {TEXT_INDEX_NAME} (plain — sync with ctx_ddl.sync_index)")
    return True

_ensure_text_index()

# CONTAINS() takes an Oracle Text *expression*, not free text: "transaction amount in cents" is
# a phrase (those words adjacent, in that order), which almost never matches a document. OR the
# terms instead, so a row matching any of them is scored by how many/how often it matches.
_STOPWORDS = {"the", "and", "for", "what", "which", "how", "does", "with", "that", "this", "are",
              "was", "were", "you", "your", "from", "into", "its", "their", "than", "then"}


def _text_query(user_query: str) -> str:
    """Natural language -> Oracle Text expression: {term} OR {term} ... (braces escape the term)."""
    terms = [t.lower() for t in re.findall(r"[A-Za-z0-9]+", user_query or "")]
    terms = [t for t in dict.fromkeys(terms) if len(t) > 2 and t not in _STOPWORDS]
    return " OR ".join(f"{{{t}}}" for t in terms) or (user_query or "")


def keyword_search_memories(query, k=5):
    sql = f"""
        SELECT m.record_id, m.metadata,
               DBMS_LOB.SUBSTR(m.content, 4000, 1) AS content,
               SCORE(1) AS score_txt
          FROM {MEMORY_TABLE} m
         WHERE CONTAINS(m.content, :kw, 1) > 0
           AND m.user_id = :u AND m.agent_id = :a
           AND (JSON_VALUE(m.metadata, '$.kind') IS NULL
                OR JSON_VALUE(m.metadata, '$.kind') <> 'tool_output')
         ORDER BY SCORE(1) DESC
         FETCH FIRST :k ROWS ONLY
    """
    with agent_conn.cursor() as cur:
        try:
            cur.execute(sql, kw=_text_query(query), u=USER_ID, a=AGENT_ID, k=k)
        except oracledb.DatabaseError as e:
            # DRG-10599 = no CTXSYS.CONTEXT index on the memory body yet (the 3.3a cell above
            # creates it). Create it on the spot and retry once, so cell order can't break the
            # three-way probe.
            if "DRG-10599" not in str(e):
                raise
            cur.execute(f"CREATE INDEX {TEXT_INDEX_NAME} ON {MEMORY_TABLE}(content) "
                        f"INDEXTYPE IS CTXSYS.CONTEXT PARAMETERS ('SYNC (ON COMMIT)')")
            cur.execute(sql, kw=_text_query(query), u=USER_ID, a=AGENT_ID, k=k)
        rows = []
        for record_id, metadata, content, score in cur:
            meta = metadata or {}
            if hasattr(content, "read"):
                content = content.read()
            rows.append({"record_id": record_id,
                         "kind": meta.get("kind", "memory"),
                         "subject": meta.get("subject", ""),
                         "content": str(content or "")[:500],
                         "score_txt": float(score) if score is not None else 0.0})
    return rows


## 3.4 Implement `hybrid_rrf_search_memories`

**TODO 5.** Fuse the keyword leg (3.3) with the vector leg (Part 2) in **one SQL statement** using Reciprocal Rank Fusion: rows in both lists outrank rows in one, and cosine distance and Oracle Text's BM25 never have to be normalised.

> 📖 See: [Part 3 guide → TODO 5](docs/part-3-retrieval.md#todo-5-implement-hybrid_rrf_search_memories)


In [ ]:
# TODO 5 — implement hybrid_rrf_search_memories(query, k=5, per_list=30, rrf_k=60).
#
# This is the single SQL that fuses cosine + keyword search. The pattern:
#
#   WITH q_emb AS (SELECT TO_VECTOR(VECTOR_EMBEDDING(<embedder> USING :q AS DATA),
#                                   384, FLOAT64) AS emb FROM dual),
#        vec   AS (-- ROW_NUMBER() OVER (ORDER BY VECTOR_DISTANCE(...)) AS r_vec
#                  --   FROM <table> JOIN <chunks> CROSS JOIN q_emb
#                  --   WHERE user_id = :u AND agent_id = :a
#                  --     AND kind <> 'tool_output'
#                  --   FETCH FIRST :n ROWS ONLY),
#        txt   AS (-- ROW_NUMBER() OVER (ORDER BY SCORE(1) DESC) AS r_txt
#                  --   FROM <table> WHERE CONTAINS(content, :kw, 1) > 0
#                  --     AND user_id = :u AND agent_id = :a
#                  --     AND kind <> 'tool_output'
#                  --   FETCH FIRST :n ROWS ONLY)
#   SELECT COALESCE(v.record_id, t.record_id),
#          ...,
#          1.0/(:rrf_k + NVL(v.r_vec, 999999))
#        + 1.0/(:rrf_k + NVL(t.r_txt, 999999))  AS rrf_score
#     FROM vec v FULL OUTER JOIN txt t ON v.record_id = t.record_id
#    ORDER BY rrf_score DESC FETCH FIRST :k ROWS ONLY;
#
# Two CTEs + FULL OUTER JOIN: each leg ranks independently, and a row in BOTH lists
# outranks a row in one — no score normalisation between vector distance and BM25.
#
# Return a list of dicts with keys: record_id, kind, subject, content (<=500
# chars), r_vec (int), r_txt (int), rrf_score (float). Use rank 999999 for the
# leg that didn't return the row (matches the NVL in the SQL).
#
# `kw` must be an Oracle Text *expression*, not the raw question: reuse `_text_query(query)`
# from §3.3. (Handing CONTAINS the whole sentence — quoted or not — makes it a phrase search
# that matches nothing, and every r_txt comes back as the 999999 sentinel.)
#
# 📖 docs/part-3-retrieval.md → TODO 5

MEMORY_TABLE = "eda_onnx_memory"

def hybrid_rrf_search_memories(query, k=5, per_list=30, rrf_k=60):
    # YOUR CODE HERE
    raise NotImplementedError("TODO 5: hybrid_rrf_search_memories")


In [ ]:
# Hard-stop checkpoint: TODO 5
_h = hybrid_rrf_search_memories("amount_cents transaction amounts stored in USD CENTS, never dollars", k=3)
assert isinstance(_h, list) and len(_h) >= 1, "❌ TODO 5: expected non-empty list"
_keys = {"record_id", "kind", "subject", "content", "r_vec", "r_txt", "rrf_score"}
assert _keys.issubset(_h[0].keys()), f"❌ TODO 5: missing keys {_keys - _h[0].keys()}"
assert all(isinstance(r["r_vec"], int) and isinstance(r["r_txt"], int) for r in _h), \
    "❌ TODO 5: r_vec / r_txt must be ints"
print("✅ TODO 5 passed — hybrid_rrf_search_memories returns RRF-fused results.")
print(f"  top result: rrf={_h[0]['rrf_score']:.4f}  r_vec={_h[0]['r_vec']}  r_txt={_h[0]['r_txt']}")


## Concept check — retrieval without vectors

A substring (`LIKE`) lookup cannot answer a question that shares no words with the stored fact. Below: the naive baseline, then `retrieve_knowledge()` from **TODO 4** — same question, one of them finds `FINANCE.TRANSACTIONS.AMOUNT_CENTS`.

<!-- workshop:concept-demo:similarity -->


In [ ]:
# ============================================================================
# WITHOUT similarity search vs WITH it — the functions from TODO 4 / 3.4
# ============================================================================
q = "how are transaction amounts stored — dollars or cents?"

# ---- WITHOUT: a plain substring lookup over the stored memory bodies -------
print("WITHOUT vector search — LIKE over the memory body:")
with agent_conn.cursor() as cur:
    cur.execute(
        f"SELECT COUNT(*) FROM {MEMORY_TABLE} "
        f" WHERE LOWER(DBMS_LOB.SUBSTR(content, 4000, 1)) LIKE '%' || LOWER(:q) || '%'",
        q="transaction amounts stored dollars cents",
    )
    print("   rows matching that exact phrase:", cur.fetchone()[0])
print("   -> ~0 rows. The fact is stored as 'amounts are stored in USD CENTS'; a natural")
print("      question shares no long substring with it. Lexical search only wins on tokens.")

# ---- WITH: the similarity search you implemented (TODO 4) -----------------
print("\nWITH similarity search — retrieve_knowledge() (vector + rerank):")
for h in retrieve_knowledge(q, k=3):
    d = h.get("distance")
    tag = f"distance={float(d):.4f}  " if d is not None else ""
    print(f"   {tag}{h['subject']}")
print("   -> the right fact surfaces with zero shared words, because the match is semantic.")


## 3.5 Three-way probe (just run)

Same query through all three retrievers. Watch the `r_vec` / `r_txt` columns — rows in both lists get the highest combined RRF score.

In [ ]:
probe_q = "amount_cents transaction amounts stored in USD CENTS, never dollars"

print("=" * 80)
print(f"QUERY: {probe_q!r}")
print("=" * 80)

print("\n--- A) VECTOR + RERANK ---")
for h in retrieve_knowledge(probe_q, k=3):
    print(f"  [{h['kind']:12s}] {h['subject'][:40]:40s}  {h['body'][:100]}")

print("\n--- B) KEYWORD ONLY ---")
for h in keyword_search_memories(probe_q, k=3):
    print(f"  [{h['kind']:12s}] score={h['score_txt']:6.2f}  {h['subject'][:40]}")

print("\n--- C) HYBRID via RRF ---")
for h in hybrid_rrf_search_memories(probe_q, k=3):
    print(f"  rrf={h['rrf_score']:.4f}  r_vec={h['r_vec']:>3}  r_txt={h['r_txt']:>3}  {h['subject'][:40]}")

# Part 6 — Toolbox & Skillbox

> 📖 Guide: [`docs/part-6-tools-and-skills.md`](docs/part-6-tools-and-skills.md) · 🔧 **TODOs 6–8** — `retrieve_tools`, `tool_run_sql`, `tool_list_skills`

The dispatch layer. A **tool** is a Python callable plus a row in `toolbox` (name, description, parameters, embedding), retrieved top-k per turn so the registry can grow without growing the prompt. A **skill** is a markdown playbook: the model always sees name + description, and reads the body on demand with `load_skill(name)`.

![Toolbox flow — register-time vs per-turn retrieval](images/cover-toolbox-flow.png)

Both tables already exist and the skillbox is seeded from [`oracle/skills`](https://github.com/oracle/skills). Below: the `@register` decorator, the `TOOLS` registry, and the core tools.


## 6.1 The `@register` decorator

One decorator does three things: introspects the function into an OpenAI-style JSON schema (what the LLM sees), registers it in the in-process `TOOLS` dict (dispatch by name), and writes an embedded row into `toolbox` (`VECTOR_EMBEDDING`, HNSW-indexed).

That third step is what keeps the prompt small: `retrieve_tools(query, k=6)` cosine-searches `toolbox`, reranks, and merges a small always-on set (`run_sql`, `search_knowledge`, `remember`, `load_skill`). The model only sees the subset this turn needs. **TODO 6:** implement `retrieve_tools`.

*(HNSW — Hierarchical Navigable Small World — is Oracle's approximate nearest-neighbour index; it lives in the `vector_memory_size` pool and scales as `log(N)`.)*


In [ ]:
import inspect, typing

TOOLS = {}                                          # name -> (callable, openai_schema)
ALWAYS_ON_TOOLS = {"search_knowledge", "run_sql", "remember"}

_PRIMS = {int: "integer", float: "number", bool: "boolean", str: "string"}


def _hint_to_json(hint):
    origin = typing.get_origin(hint)
    if hint in _PRIMS:
        return {"type": _PRIMS[hint]}
    if origin in (list, typing.List):
        args = typing.get_args(hint) or (str,)
        return {"type": "array", "items": _hint_to_json(args[0])}
    if origin in (dict, typing.Dict):
        return {"type": "object"}
    if origin is typing.Union:
        non_none = [a for a in typing.get_args(hint) if a is not type(None)]
        if len(non_none) == 1:
            return _hint_to_json(non_none[0])
    return {"type": "string"}


def _build_schema(fn):
    raw_name = fn.__name__
    name = raw_name[5:] if raw_name.startswith("tool_") else raw_name
    description = (inspect.getdoc(fn) or "").strip()
    if not description:
        raise ValueError(f"tool {name!r} has no docstring; @register needs one for retrieval")
    sig = inspect.signature(fn)
    hints = typing.get_type_hints(fn)
    properties, required = {}, []
    for pname, param in sig.parameters.items():
        prop = _hint_to_json(hints.get(pname, str))
        if param.default is not inspect.Parameter.empty and param.default is not None:
            prop["default"] = param.default
        else:
            required.append(pname)
        properties[pname] = prop
    parameters = {"type": "object", "properties": properties, "required": required}
    return name, description, parameters, {
        "type": "function",
        "function": {"name": name, "description": description, "parameters": parameters},
    }



In [ ]:
def register(fn):
    name, description, parameters, openai_schema = _build_schema(fn)
    TOOLS[name] = (fn, openai_schema)
    arg_text = " ".join(parameters["properties"].keys())
    embed_text = f"{name}: {description}\nargs: {arg_text}"
    with agent_conn.cursor() as cur:
        cur.execute(
            "MERGE INTO toolbox t USING (SELECT :tn AS n FROM dual) s ON (t.name = s.n) "
            "WHEN MATCHED THEN UPDATE SET description = :td, parameters = :tp, "
            f" embedding = VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :etext AS DATA), "
            " updated_at = CURRENT_TIMESTAMP "
            "WHEN NOT MATCHED THEN INSERT (name, description, parameters, embedding) "
            f" VALUES (:tn, :td, :tp, VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :etext AS DATA))",
            tn=name, td=description, tp=json.dumps(parameters), etext=embed_text,
        )
    agent_conn.commit()
    return fn

In [ ]:
def retrieve_tools(query, k=6):
    # TODO 6: make the toolbox retrievable by meaning. Cosine-rank `toolbox` (HNSW)
    # for the query, rerank the shortlist, then union in ALWAYS_ON_TOOLS.
    # 📖 docs/part-6-tools-and-skills.md
    raise NotImplementedError("TODO 6: retrieve_tools")


## 6.2 Register `tool_run_sql`

The agent's way to query live data, and it must be **read-only**: `SELECT` and `WITH` only.

1. Reject anything that doesn't match `_READ_ONLY`.
2. Execute on `agent_conn`.
3. Return up to `max_rows` as JSON: `{"columns": [...], "rows": [...], "row_count": N}` — reading CLOB cells with `.read()`.
4. Return `{"error": "..."}` on any database error.

The docstring is the tool description the LLM sees — write it for the model.

> 📖 See: [Part 6 guide → TODO 7](docs/part-6-tools-and-skills.md#todo-7-register-tool_run_sql)


In [ ]:
_READ_ONLY = re.compile(r"^\s*(select|with)\b", re.IGNORECASE)


# TODO 7 — decorate with @register and implement.
# Requirements:
#   - signature: tool_run_sql(sql: str, max_rows: int = 50) -> str
#   - reject any statement that doesn't match _READ_ONLY
#   - return JSON: {"columns": [...], "rows": [...], "row_count": N} or {"error": "..."}
#   - handle CLOB values: v.read() if hasattr(v, "read") else v
#   - the docstring is what the LLM sees as the tool description — make it descriptive
# 📖 docs/part-6-tools-and-skills.md

@register
def tool_run_sql(sql: str, max_rows: int = 50) -> str:
    """Execute a READ-ONLY SQL statement (SELECT/WITH only) against the target Oracle AI Database
    and return up to `max_rows` rows as JSON. Reject any statement that isn't read-only.
    """
    # YOUR CODE HERE
    raise NotImplementedError("TODO 7: tool_run_sql")


In [ ]:
# Hard-stop checkpoints: TODO 6 (retrieve_tools) and TODO 7 (tool_run_sql).
assert "run_sql" in TOOLS, "❌ TODO 7: tool_run_sql not registered. Did you @register it?"
_out = json.loads(tool_run_sql("SELECT 1 AS one FROM dual"))
assert _out.get("row_count") == 1, "❌ TODO 7: SELECT 1 didn't return one row."
_bad = json.loads(tool_run_sql("DROP TABLE x"))
assert "error" in _bad, "❌ TODO 7: read-only guard let a DROP through!"
print("✅ TODO 7 passed: tool_run_sql registered and read-only enforced.")
_retrieved_names = sorted(s["function"]["name"]
                          for s in retrieve_tools("query the bank with read-only SQL", k=6))
assert "run_sql" in _retrieved_names, "❌ TODO 6: retrieve_tools did not find run_sql."
print(f"✅ TODO 6 passed: retrieve_tools returned {_retrieved_names}.")

## 6.3 The core toolset

Pre-built core tools that round out the harness: `scan_database`, `search_knowledge`, and `remember`. The running Flask app exposes additional live tools (scratch storage, MLE compute, duality views, spatial, automations) — see [`app/README.md`](app/README.md) and the advanced guides in `docs/`.


In [ ]:
@register
def tool_scan_database(owner: str) -> str:
    """Scan the specified schema and update institutional knowledge.
    Run this when the user asks about a schema you have never seen or when the knowledge store is stale.
    """
    return json.dumps(run_scan(agent_conn, owner=owner))


@register
def tool_search_knowledge(query: str, k: int = 5, kinds: list[str] | None = None) -> str:
    """Search institutional knowledge by semantic similarity.
    Use this BEFORE running SQL to discover relevant tables and columns.
    """
    hits = retrieve_knowledge(query, k=k, kinds=kinds)
    for h in hits:
        h["body"] = h["body"][:500]
    return json.dumps(hits)


@register
def tool_remember(subject: str, body: str, kind: str = "correction") -> str:
    """Persist a correction or learning so future turns benefit.
    Use when the user corrects you or you discover a non-obvious fact.
    """
    fact = Fact(kind=kind, subject=subject, body=body, metadata={"source": "agent_remember"})
    new, updated, _ = write_facts([fact])
    return json.dumps({"ok": True, "new": new, "updated": updated})


print(f"registered {len(TOOLS)} core tools: {sorted(TOOLS)}")


## 6.4 Skill manifest

The skillbox is pre-populated. `build_skill_manifest(query, k=3)` returns the top-k skill names + descriptions, formatted as a one-line-per-skill block prepended to every turn's context. The model can `load_skill(name)` to retrieve the full body. **Your job (TODO 8):** implement `tool_list_skills`, the semantic search behind skill discovery.

In [ ]:
@register
def tool_load_skill(name: str) -> str:
    """Load the full content of a named skill from the skillbox.
    Use this when the system prompt's manifest lists a skill relevant to the current task.
    `name` is the full namespace, e.g. "agent/schema-discovery".
    """
    with agent_conn.cursor() as cur:
        cur.execute("SELECT description, body, source_url, category FROM skillbox WHERE name = :n",
                    n=name)
        row = cur.fetchone()
    if not row:
        return json.dumps({"error": f"no skill named {name!r}"})
    desc, body, url, category = row
    body_text = body.read() if hasattr(body, "read") else str(body or "")
    return json.dumps({"name": name, "category": category, "description": desc,
                       "source_url": url, "body": body_text})


@register
def tool_list_skills(query: str, k: int = 5) -> str:
    """Search the skillbox semantically. Returns top-k skills (name + description)."""
    # TODO 8: rank skills by cosine distance to the query (HNSW over skillbox)
    # and return a JSON list of {name, category, description}. load_skill above
    # is the on-demand half; this is the discovery half.
    # 📖 docs/part-6-tools-and-skills.md
    raise NotImplementedError("TODO 8: tool_list_skills")


ALWAYS_ON_TOOLS.add("load_skill")


def build_skill_manifest(query, k=3):
    try:
        with agent_conn.cursor() as cur:
            cur.execute(
                "SELECT name, description FROM skillbox "
                f" ORDER BY VECTOR_DISTANCE(embedding, VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :q AS DATA), COSINE) "
                " FETCH FIRST :k ROWS ONLY", q=query, k=k)
            rows = list(cur)
    except oracledb.DatabaseError:
        return ""
    if not rows:
        return ""
    lines = [f"  - {n} — {(d.read() if hasattr(d, 'read') else d)[:240]}" for n, d in rows]
    return ("Available skills (call load_skill(name) to read the full guide and follow it):\n"
            + "\n".join(lines) + "\n\n")


print(f"registered: load_skill, list_skills (TOOLS total: {len(TOOLS)})")

# Hard-stop checkpoint: TODO 8. Runs after the skillbox is populated so the search
# has real rows to find; an empty skillbox degrades to a shape check.
_skill_hits = json.loads(tool_list_skills("discover the schema of an Oracle database", k=3))
assert isinstance(_skill_hits, list), "❌ TODO 8: tool_list_skills must return a JSON array."
with agent_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM skillbox")
    _skill_count = cur.fetchone()[0]
if _skill_count:
    assert _skill_hits, "❌ TODO 8: skillbox has rows but the semantic search returned none."
    assert {"name", "category", "description"} <= set(_skill_hits[0]), \
        "❌ TODO 8: each hit needs name, category, and description."
print(f"✅ TODO 8 passed: skillbox search returned {len(_skill_hits)} of {_skill_count} skills.")


## 6.5 Driving the globe — `focus_world` *(run it — not a TODO)*

Parts 1–6 let the agent *read* the bank; this lets it *point at* it. The app's World Explorer is a 3D globe of branches, merchants, and flagged activity, fed by `GET /api/world`; `focus_world` resolves a name to `(lat, lng)` and emits a Socket.IO event that flies the camera.

A notebook has no browser socket, so the cell below registers the **resolution half** — the same SQL the app runs — and returns the anchor as JSON. Everything except the final emit is identical to the app's tool: a tool is SQL plus a payload shape.

| `target_kind` | Resolves against | Example |
|---|---|---|
| `branch` | branch code, name, or city | `Wall Street`, `Dubai` |
| `merchant` | merchant name or category | `BitVault Exchange` |
| `customer` | customer full name, anchored at their oldest account's branch | `Isabella Allen` |
| `region` | one of the four bank regions | `EUROPE` |

> 📖 See: [`app/README.md`](app/README.md) · [`docs/part-6-tools-and-skills.md`](docs/part-6-tools-and-skills.md)


In [ ]:
# --- Globe tool: resolve a name to a place -----------------------------------
#
# The app's World Explorer globe is fed by GET /api/world (branches, merchants,
# flagged transactions). The agent drives the camera through `focus_world`:
# the Flask backend resolves the target and emits `focus_world` over Socket.IO.
# A notebook has no socket, so we register the *resolution* half and return the
# anchor as JSON. Everything but the final emit matches app/backend/agent/tools.py.

_REGION_CENTROIDS = {
    "AMERICAS":     (38.0,  -98.0, 2.6),
    "EUROPE":       (50.0,    8.0, 2.6),
    "MIDDLE_EAST":  (27.0,   45.0, 2.4),
    "ASIA_PACIFIC": (12.0,  105.0, 2.6),
}


def _resolve_world_target(kind, target):
    """Resolve a globe target (branch / merchant / customer / region) to a
    {lat, lng, label} anchor. Returns None when nothing matches."""
    target = target.strip()
    needle = f"%{target.lower()}%"
    exact = target.lower()

    if kind == "region":
        c = _REGION_CENTROIDS.get(target.upper())
        if not c:
            return None
        return {"lat": c[0], "lng": c[1], "label": target.upper(),
                "region": target.upper(), "metadata": {"altitude_hint": c[2]}}

    with agent_conn.cursor() as cur:
        if kind == "branch":
            cur.execute(
                f"SELECT branch_id, name, city, country, region, latitude, longitude "
                f"  FROM {DEMO_USER}.branches "
                f" WHERE LOWER(branch_code) = :exact OR LOWER(name) LIKE :needle "
                f"    OR LOWER(city) LIKE :needle "
                f" ORDER BY CASE WHEN LOWER(branch_code) = :exact THEN 0 ELSE 1 END "
                f" FETCH FIRST 1 ROWS ONLY", exact=exact, needle=needle)
            row = cur.fetchone()
            if not row:
                return None
            bid, name, city, country, region, lat, lng = row
            return {"lat": float(lat), "lng": float(lng),
                    "label": f"{name} ({city})", "region": region,
                    "metadata": {"branch_id": int(bid), "country": country}}

        if kind == "merchant":
            cur.execute(
                f"SELECT merchant_id, name, category, country, region, latitude, longitude "
                f"  FROM {DEMO_USER}.merchants "
                f" WHERE LOWER(name) LIKE :needle OR LOWER(category) LIKE :needle "
                f" FETCH FIRST 1 ROWS ONLY", needle=needle)
            row = cur.fetchone()
            if not row:
                return None
            mid, name, category, country, region, lat, lng = row
            return {"lat": float(lat), "lng": float(lng),
                    "label": f"{name} ({category})", "region": region,
                    "metadata": {"merchant_id": int(mid), "country": country}}

        if kind == "customer":
            # Anchor the customer at their primary (oldest) account's branch.
            cur.execute(
                f"SELECT cu.customer_id, cu.full_name, b.name, b.city, b.region, "
                f"       b.latitude, b.longitude "
                f"  FROM {DEMO_USER}.customers cu "
                f"  JOIN {DEMO_USER}.accounts a ON a.customer_id = cu.customer_id "
                f"  JOIN {DEMO_USER}.branches b ON b.branch_id = a.branch_id "
                f" WHERE LOWER(cu.full_name) LIKE :needle "
                f" ORDER BY a.opened_ts "
                f" FETCH FIRST 1 ROWS ONLY", needle=needle)
            row = cur.fetchone()
            if not row:
                return None
            cid, name, branch_name, city, region, lat, lng = row
            return {"lat": float(lat), "lng": float(lng),
                    "label": f"{name} -> {branch_name} ({city})", "region": region,
                    "metadata": {"customer_id": int(cid), "branch": branch_name}}

    return None


@register
def tool_focus_world(target_kind: str, target: str, altitude: float = 1.5) -> str:
    """Drive the World Explorer globe - fly the camera to a branch, merchant, customer, or bank region.
    Use this whenever the user says things like "show me the Wall Street branch on the globe",
    "fly to Dubai", or "zoom to Europe".
    target_kind is one of: branch, merchant, customer, region.
    target is the entity name / code / region label, e.g. 'Wall Street', 'BitVault Exchange', 'Isabella Allen', 'EUROPE'.
    altitude is the globe camera altitude (1.0 = close, 2.5 = global view).
    """
    kind = (target_kind or "").strip().lower()
    if kind not in ("branch", "merchant", "customer", "region"):
        return json.dumps({"error": f"unknown target_kind {target_kind!r}; "
                                   "must be one of branch|merchant|customer|region"})
    if not (target or "").strip():
        return json.dumps({"error": "target is required"})
    try:
        anchor = _resolve_world_target(kind, target)
    except Exception as e:
        return json.dumps({"error": f"resolve failed: {type(e).__name__}: {e}"})
    if anchor is None:
        return json.dumps({"error": f"no {kind} found matching {target!r}"})

    payload = {
        "kind": kind,
        "target": target,
        "lat": anchor["lat"],
        "lng": anchor["lng"],
        "altitude": max(0.6, min(float(altitude or 1.5), 4.0)),
        "label": anchor.get("label", target),
        "region": anchor.get("region"),
        "metadata": anchor.get("metadata", {}),
    }
    # In the app this is where the Socket.IO `focus_world` emit happens.
    return json.dumps({"ok": True, **payload})


print(f"registered {len(TOOLS)} tools (globe tool included): "
      f"{'focus_world' in TOOLS}")


In [ ]:
# Fly the camera around the bank. Each call returns the anchor the app would
# emit to the WorldExplorer; the last one shows the not-found path.
for kind, target in [("branch",   "Wall Street"),
                     ("branch",   "Dubai"),
                     ("merchant", "BitVault Exchange"),
                     ("customer", "Isabella Allen"),
                     ("region",   "MIDDLE_EAST"),
                     ("branch",   "No Such Branch")]:
    out = json.loads(tool_focus_world(kind, target))
    if out.get("ok"):
        print(f"  {kind:9s} {target:20s} -> {out['lat']:8.4f}, {out['lng']:9.4f}  {out['label']}")
    else:
        print(f"  {kind:9s} {target:20s} -> {out['error']}")


# Part 7 — Context Engineering & the Agent Loop

> 📖 Guide: [`docs/part-7-agent-loop.md`](docs/part-7-agent-loop.md) · 🔧 **TODO 9** — `agent_turn`

*The keystone.* Parts 2–6 are primitives that each stand on their own; Part 7 is the loop that ties them together — assemble context, call the LLM with the retrieved tools, dispatch what it asks for, append the result, repeat. Roughly 90 lines of Python, and worth reading twice.

```
build_context → call LLM with retrieved tools → if tool_calls: dispatch  else: final → log
```


## 7.1 `SYSTEM_PROMPT` + `build_context` + `log_message`

1. **`SYSTEM_PROMPT`** — the agent's job description and constraints; every rule shapes a real choice (the USD-cents rule stops it reporting `amount_cents` as dollars).
2. **`get_thread(thread_id)`** — cached OAMP thread objects: messages, rolling summary, context card.
3. **`build_context(thread_id, user_query)`** — skill manifest + context card + retrieved knowledge + the question, selected, ordered, and compressed for the model.
4. **`log_message`** — fire-and-forget persistence of each turn to OAMP.

TODO 9 is the loop that uses them.


In [ ]:
SYSTEM_PROMPT = (
    "You are the Meridian Bank Financial Data Agent — an AML / financial-crime analyst harness operating against an Oracle AI Database.\n"
    "\n"
    "Your job is to answer natural-language questions about the data by reasoning over:\n"
    "  - the institutional knowledge you have accumulated about the database;\n"
    "  - the live FINANCE schema, via read-only SQL when you need runtime facts.\n"
    "\n"
    "How to work:\n"
    "  1. ALWAYS call search_knowledge first with a paraphrase of the user's question.\n"
    "  2. If the user references a schema you have no facts about, call scan_database, then search again.\n"
    "  3. Keep SQL read-only. Use run_sql only; never attempt DDL or DML through it.\n"
    "  4. Remember that transactions.amount_cents and accounts.balance_cents are USD CENTS —\n"
    "     divide by 100 for dollars, and always state which unit you are reporting.\n"
    "  5. When you discover a non-obvious fact or the user corrects you, call remember\n"
    "     so future turns benefit.\n"
    "  6. Prefer short, direct answers. Quote table and column names verbatim.\n"
    "  7. If a tool fails, read the error, adjust, and try once more — don't spin.\n"
    "\n"
    "Never fabricate a table or column. If you're unsure, say so and propose a scan."
)


THREADS = {}


In [ ]:
import threading

# OAMP keeps one store bound to one connection and is not thread-safe, but log_message() below
# writes from a ThreadPoolExecutor. Every OAMP call therefore goes through this lock.
# Re-entrant: get_thread() takes it too, and _do_log() already holds it on that thread.
_OAMP_LOCK = threading.RLock()

_MEMORY_CFG = MemoryExtractionConfig(
    extract_memories=True,
    memory_extraction_frequency=2,
    memory_extraction_window=4,
    enable_context_summary=True,
    context_summary_update_frequency=4,
)


def get_thread(thread_id):
    with _OAMP_LOCK:
        if thread_id in THREADS:
            return THREADS[thread_id]
        try:
            thread = memory_client.get_thread(thread_id)
        except Exception:
            try:
                thread = memory_client.create_thread(
                    thread_id=thread_id, user_id=USER_ID, agent_id=AGENT_ID,
                    memory_extraction_config=_MEMORY_CFG,
                )
            except ValueError:
                # "Thread already exists." — a concurrent caller won the race, or the row survived
                # a previous run of this notebook. Adopt what is there instead of failing.
                thread = memory_client.get_thread(thread_id)
        THREADS[thread_id] = thread
        return thread


In [ ]:

def build_context(thread_id, user_query, k_knowledge=3):
    parts = []
    try:
        manifest = build_skill_manifest(user_query, k=3)
    except NameError:
        manifest = ""
    if manifest:
        parts.append(manifest.rstrip())

    thread = get_thread(thread_id)
    try:
        card = thread.get_context_card()
    except Exception as e:
        # OAMP builds the context card by asking the LLM to summarise the thread, so a transient
        # rate limit / quota exhaustion / network blip lands here. Degrade instead of failing the
        # turn: the rest of the context block (manifest + retrieved facts) is still valid.
        card = None
        print(f"  ! OAMP context card unavailable ({type(e).__name__}) — continuing without it")
    card_text = str(card) if card else ""
    if card_text:
        parts.append("## Memory context (from OAMP)")
        parts.append(card_text)

    hits = retrieve_knowledge(user_query, k=k_knowledge)
    if hits:
        parts.append("\n## Institutional knowledge (top matches)")
        for h in hits:
            parts.append(f"- ({h['kind']}) {h['subject']} - {h['body'][:280]}")

    parts.append("\n## User question")
    parts.append(user_query)
    return "\n".join(parts)

## Concept check — context summarization

`build_context()` is half the story: OAMP also keeps a **rolling context card** per thread (`enable_context_summary=True`, refreshed every `context_summary_update_frequency` turns), so a long thread does not replay its raw transcript every turn. Below: four corrections on a throwaway thread, raw transcript vs generated card.

<!-- workshop:concept-demo:context -->


In [ ]:
# ============================================================================
# WITHOUT context summarization vs WITH it — OAMP's context card
# ============================================================================
from oracleagentmemory.apis.thread import Message

probe_thread = "context-demo"
_corrections = [
    "Remember: transactions.amount_cents is USD CENTS, never dollars.",
    "Also: customers.risk_rating is a 1-100 score, higher means riskier.",
    "Third: branches.region is one of AMERICAS, EUROPE, MIDDLE_EAST, ASIA_PACIFIC.",
    "Fourth: SAR_REPORTS is compliance-only and forbidden to analysts.",
]
get_thread(probe_thread).add_messages([Message(role="user", content=m) for m in _corrections])

_raw = "\n".join(m.content for m in get_thread(probe_thread).get_messages())
try:
    _card = str(get_thread(probe_thread).get_context_card() or "")
except Exception as e:
    _card = ""
    print("  ! OAMP context card unavailable:", type(e).__name__)

print("WITHOUT the context card, every turn replays the raw transcript:")
print(f"   raw transcript: {len(_raw)} chars across {len(_corrections)} messages")
print("   -> unbounded growth; older corrections eventually fall outside the window.")

print("\nWITH OAMP's context card (enable_context_summary=True, refreshed every 4 turns):")
print(f"   card: {len(_card)} chars — a structured, bounded summary")
if _card:
    print("   preview:", _card[:280].replace(chr(10), " "))
if _card and _raw:
    print(f"   right now raw:card = {len(_raw) / max(len(_card), 1):.2f}x — with only "
          f"{len(_corrections)} messages the card is larger;")
    print("   append 100 turns and the transcript grows linearly while the card stays ~constant.")
print("\n-> The card is a rolling, model-generated summary of topics + decisions. It is what")
print("   lets a long thread stay inside the context window without losing what was decided.")


In [ ]:
_LOG_EXECUTOR = concurrent.futures.ThreadPoolExecutor(max_workers=4, thread_name_prefix="oamp-log")
_LOG_PENDING = []

In [ ]:


def log_message(thread_id, role, content):
    from oracleagentmemory.apis.thread import Message
    def _do_log():
        with _OAMP_LOCK:                      # OAMP: never call it from two threads at once
            get_thread(thread_id).add_messages([Message(role=role, content=content)])
    still = [f for f in _LOG_PENDING if not f.done()]
    _LOG_PENDING[:] = still
    _LOG_PENDING.append(_LOG_EXECUTOR.submit(_do_log))


print("SYSTEM_PROMPT + get_thread + build_context + log_message ready")

## 7.2 Implement `agent_turn`

The heart of the harness. Loop: build context → call LLM with retrieved tools → if `tool_calls` dispatch and append results → else break with final answer. Guard with iteration cap and wall-clock budget. If both run out without a final answer, ask one more time without tools (forced final).

> 📖 **See:** [Part 7 guide → TODO 9](docs/part-7-agent-loop.md#todo-9-implement-agent_turn)


In [ ]:
# TODO 9 — implement agent_turn(user_query, thread_id, max_iterations, budget_seconds, verbose)
# 📖 docs/part-7-agent-loop.md — read the dispatch pattern there before you start.

def agent_turn(user_query, thread_id="default", max_iterations=8, budget_seconds=360.0, verbose=True):
    started = time.time()
    log_message(thread_id, "user", user_query)

    context = build_context(thread_id, user_query)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": context},
    ]
    tool_schemas = retrieve_tools(user_query, k=6)

    final = ""
    step = 0
    for step in range(max_iterations):
        if time.time() - started > budget_seconds:
            if verbose: print(f"  ! budget exhausted at iteration {step}")
            break

        # YOUR CODE HERE:
        # 1. resp = chat(messages, tools=tool_schemas)
        # 2. msg = resp.choices[0].message
        # 3. If no tool_calls, set final = msg.content and break.
        # 4. Otherwise:
        #    a. Append the assistant tool_calls message (echo back so each tool result has its matching call).
        #    b. For each tool_call, dispatch from TOOLS, append {"role": "tool", "tool_call_id": ..., "content": output}.
        raise NotImplementedError("TODO 9: agent_turn dispatch loop")

    if not final:
        messages.append({"role": "user",
                         "content": "Budget exhausted. Provide your best answer now, no more tools."})
        resp = chat(messages, tools=None)
        final = resp.choices[0].message.content or "(no answer produced)"

    log_message(thread_id, "assistant", final)
    try:
        with _OAMP_LOCK:
            memory_client.add_memory(
                f"User: {user_query}\n\nAssistant: {final}",
                user_id=USER_ID, agent_id=AGENT_ID,
                thread_id=thread_id,
                metadata={"kind": "episodic", "thread_id": thread_id,
                          "user_query": user_query[:240]},
            )
    except Exception:
        pass

    if verbose: print(f"  [{time.time() - started:.1f}s, {step + 1} steps]")
    return final


In [ ]:
# Hard-stop checkpoint: TODO 9
assert callable(agent_turn), "❌ TODO 9: agent_turn not defined."
import inspect as _ins
_sig = _ins.signature(agent_turn)
for _p in ("user_query", "thread_id", "max_iterations", "budget_seconds"):
    assert _p in _sig.parameters, f"❌ TODO 9: agent_turn missing parameter {_p!r}."
print("✅ TODO 9 passed — agent_turn signature looks right. Demos below will exercise it.")

## 7.3 Three-turn demo (just run)

Three turns on the same thread, each exercising a different harness component:

1. Discovery (forces `search_knowledge`).
2. Live data (forces `run_sql`).
3. Correction + persistence (forces `remember`).

In [ ]:
thread = "demo-session-1"

q1 = "What's in the FINANCE schema? Briefly — list the entities and how they relate."
print("USER:", q1)
print("\nASSISTANT:", agent_turn(q1, thread_id=thread))

In [ ]:
q2 = "Which branch regions have the most FLAGGED or BLOCKED transactions? Show me a small table sorted by count desc."
print("USER:", q2)
print("\nASSISTANT:", agent_turn(q2, thread_id=thread))


In [ ]:
q3 = ("Important: in the FINANCE schema, transactions.amount_cents is always USD CENTS, "
      "never dollars. And customers.risk_rating is a 1-100 score, higher means riskier. "
      "Save EACH as a separate 'correction' memory by calling remember BEFORE you respond, "
      "then confirm with the memory IDs.")
print("USER:", q3)
print("\nASSISTANT:", agent_turn(q3, thread_id=thread))


## Concept check — DBFS scratch state

Mid-task scratch space is a real problem: a Python variable is not transactional, invisible to other sessions, and gone on restart. Below: a draft in a Python variable, then the same bytes written to **Oracle DBFS** — a filesystem inside the database — and read back through a second connection.

<!-- workshop:concept-demo:dbfs -->


In [ ]:
# ============================================================================
# WITHOUT DBFS vs WITH DBFS — durable, transactional scratch state
# ============================================================================
draft = ("SELECT region, COUNT(*) FROM FINANCE.transactions "
         "WHERE status = 'FLAGGED' GROUP BY region")

# ---- WITHOUT: a Python variable -------------------------------------------
print("WITHOUT DBFS — a Python variable holds the draft:")
print("   ", draft[:70], "...")
agent_conn.rollback()   # a DB rollback cannot touch a Python variable
print("   after agent_conn.rollback(): the draft is still here (it was never in the DB).")
print("   -> not transactional with the agent's writes, invisible to any other session,")
print("      gone the moment the kernel restarts.")

# ---- WITH: Oracle DBFS — a filesystem INSIDE the database -----------------
DBFS_MOUNT = "/scratch"
DBFS_DIR   = f"{DBFS_MOUNT}/workshop"
DBFS_PATH  = f"{DBFS_DIR}/amount_rule.sql"

_MKDIR = ("DECLARE l_props DBMS_DBFS_CONTENT_PROPERTIES_T := DBMS_DBFS_CONTENT_PROPERTIES_T(); "
          "BEGIN DBMS_DBFS_CONTENT.CREATEDIRECTORY(path => :d, properties => l_props); "
          "EXCEPTION WHEN OTHERS THEN NULL; END;")
_DELETE = "BEGIN DBMS_DBFS_CONTENT.DELETEFILE(:p); EXCEPTION WHEN OTHERS THEN NULL; END;"
_CREATE = ("DECLARE l_props DBMS_DBFS_CONTENT_PROPERTIES_T := DBMS_DBFS_CONTENT_PROPERTIES_T(); "
           "l_blob BLOB := :b; BEGIN "
           "DBMS_DBFS_CONTENT.CREATEFILE(path => :p, properties => l_props, content => l_blob); "
           "COMMIT; END;")

with agent_conn.cursor() as cur:
    cur.execute(_MKDIR, d=DBFS_DIR)
    cur.execute(_DELETE, p=DBFS_PATH)
    cur.setinputsizes(b=oracledb.DB_TYPE_BLOB)
    cur.execute(_CREATE, p=DBFS_PATH, b=draft.encode())
agent_conn.commit()
print(f"\nWITH DBFS — wrote {len(draft)} bytes to {DBFS_PATH}")

# Prove it is really IN the database: read the bytes back through a BRAND-NEW
# connection (not agent_conn).
agent_conn2 = connect(AGENT_USER, AGENT_PASS, SYS_DSN)
_READ = ("DECLARE l_props DBMS_DBFS_CONTENT_PROPERTIES_T := DBMS_DBFS_CONTENT_PROPERTIES_T(); "
         "l_blob BLOB; l_item NUMBER; BEGIN "
         "DBMS_DBFS_CONTENT.GETPATH(path => :p, properties => l_props, content => l_blob, item_type => l_item); "
         ":out := l_blob; END;")
with agent_conn2.cursor() as cur:
    out = cur.var(oracledb.DB_TYPE_BLOB)
    cur.execute(_READ, p=DBFS_PATH, out=out)
    read_back = out.getvalue().read().decode()
print("   read back from a fresh connection:", read_back[:60], "...")
assert read_back == draft, "DBFS round-trip mismatch"
print("   OK — identical bytes: the draft is committed in the database, not in kernel RAM.")
print("\n-> DBFS gives the agent a real, transactional filesystem: the same ACID guarantees")
print("   as the memory tables, visible across sessions, surviving a kernel restart.")
agent_conn2.close()


<!-- workshop:part-12:begin -->

# Part 12 — The bank's morning: an autonomous AML triage

> 📖 Guide: [`docs/part-12-autonomous-aml-triage.md`](docs/part-12-autonomous-aml-triage.md) · ▶️ **no TODO — run it.** ~15 minutes, three alerts, each a full model round-trip.

Parts 1–7 built a harness that answers questions. A compliance desk does not work that way: alerts arrive on their own schedule and every one needs a decision the bank can defend. Autonomy here is four concrete properties:

| Property | Where it lives |
|---|---|
| **A trigger, not a prompt** | §12.3 builds the queue from `FINANCE` |
| **A decision with a stated reason** | §12.5–12.6 — a validated decision record |
| **A record a human can replay** | `AGENT.AML_TRIAGE` rows + `case_decision` memories |
| **A budget and a boundary** | metered model calls, ≤6 iterations / 120 s per alert, `ESCALATE` only recommends |

The agent cannot write to `FINANCE`, cannot file a SAR, and cannot see this queue at all without clearance ([Part 8](docs/part-8-deep-data-security.md)). The question §12.8 answers: if the desk ran every morning, what would it decide — and what would that be worth?

In [ ]:
# Part 12 uses the harness you built above. Fail here with one clear message instead of
# halfway through a triage run.
_PARTS_REQUIRED = ("agent_conn", "agent_turn", "chat", "retrieve_knowledge", "retrieve_tools",
                   "tool_run_sql", "write_facts", "Fact", "TOOLS")
_missing_symbols = [name for name in _PARTS_REQUIRED if name not in globals()]
assert not _missing_symbols, (
    f"❌ Part 12 needs {', '.join(_missing_symbols)} — finish the TODOs above "
    "(or open notebook_complete.ipynb) before running the capstone."
)
print("✅ harness present:", ", ".join(_PARTS_REQUIRED))

## 12.1 What an alert is — the fraud data, in one screen

| Where | What it holds | Why the agent cares |
|---|---|---|
| `FINANCE.transactions.status` | `COMPLETED` / `FLAGGED` / `BLOCKED` | the bank's own rules already fired — this *is* the queue |
| `FINANCE.transactions.flag_reason` | which AML typology fired | turns a row into an investigation |
| `FINANCE.transactions.amount_cents` | integer USD **cents** | ÷100; the seed plants this trap deliberately |
| `FINANCE.sar_reports` | what compliance already filed | prior history changes today's decision |

| `flag_reason` | Pattern the seed plants |
|---|---|
| `STRUCTURING` | deposits just under the $10,000 CTR threshold, repeated |
| `GEO_VELOCITY` | one card, two far-apart regions, hours apart |
| `HIGH_RISK_COUNTRY` | wires to elevated-risk corridors after an inbound credit |
| `RAPID_CASH_OUT` | inbound wire, then ATM withdrawals within ~48 hours |
| `LARGE_CASH_DEPOSIT` | a single cash deposit above $50,000 |

A transaction is not an alert: *"this deposit was $9,983"* is noise, while *"nine deposits between $8,000 and $9,999 in two weeks, plus a SAR under review"* is a case. §12.3 builds that customer-scoped bundle in SQL.

## 12.2 The triage ledger — the harness's own bookkeeping

Each decision lands in two places, because two audiences read it:

| Store | Shape | Who reads it |
|---|---|---|
| `AGENT.AML_TRIAGE` | one row per `(customer, typology)`: decision, confidence, rationale, evidence window | an examiner or a SQL dashboard |
| OAMP memories, `kind="case_decision"` | the same decision as a sentence | **the agent itself**, next morning |

Note what is *not* happening: no writes to `FINANCE`, ever. The write surface is the `AGENT` schema — the trust boundary from Part 1, made literal.

In [ ]:
# §12.2 — The ledger (created once) + two helpers the cells below share.
AML_LEDGER = "AGENT.AML_TRIAGE"


def _rows(sql, **binds):
    """Run SQL and return list[dict] with lowercased column names."""
    with agent_conn.cursor() as cur:
        cur.execute(sql, **binds)
        cols = [d[0].lower() for d in cur.description]
        return [dict(zip(cols, row)) for row in cur]


def _money(cents):
    return f"${(cents or 0) / 100:,.2f}"


with agent_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM user_tables WHERE table_name = 'AML_TRIAGE'")
    if not cur.fetchone()[0]:
        cur.execute("""
            CREATE TABLE aml_triage (
              customer_id    NUMBER(10),
              typology       VARCHAR2(60),
              customer_name  VARCHAR2(120),
              risk_rating    NUMBER(3),
              flagged_txns   NUMBER(6),
              exposure_cents NUMBER(15),
              window_start   TIMESTAMP,
              window_end     TIMESTAMP,
              decision       VARCHAR2(20),
              confidence     NUMBER(3,2),
              rationale      VARCHAR2(1000),
              next_action    VARCHAR2(400),
              sar_reason     VARCHAR2(60),
              decided_at     TIMESTAMP,
              decided_by     VARCHAR2(60),
              CONSTRAINT aml_triage_pk PRIMARY KEY (customer_id, typology)
            )""")
        print(f"created {AML_LEDGER}")
    else:
        print(f"{AML_LEDGER} already present")
agent_conn.commit()

_ledger_rows = _rows("SELECT COUNT(*) AS n FROM aml_triage")[0]["n"]
print(f"ledger: {_ledger_rows} decision(s) on record")
print("decision vocabulary: ESCALATE · KYC_REVIEW · DISMISS · REVIEW_REQUIRED — set by the harness, never by the model")

## 12.3 The alert queue — a trigger, not a prompt

The run starts from the data: every customer with `FLAGGED` / `BLOCKED` activity inside the lookback window, grouped by typology, ordered by risk rating and exposure. Three knobs:

- **`ALERT_LOOKBACK_DAYS`** — how far back "incoming" reaches.
- **`TRIAGE_LIMIT`** — how many alerts this run works.
- **`IGNORE_WATERMARK`** — set it to `False` and the desk only works alerts newer than the last recorded run, so a second run in the same window honestly reports **0 new alerts**.

The seeded bank has **130+ alerts**, true positives by construction, so expect an ESCALATE-heavy first run. A production queue is the mirror image — which is why a dismissal has to be written down with a reason.

In [ ]:
# §12.3 — Build the queue. This is the "incoming data" the desk reacts to.
ALERT_LOOKBACK_DAYS = 30     # window of activity this run looks at
TRIAGE_LIMIT        = 3      # alerts to work now; raise after the first pass
IGNORE_WATERMARK    = True   # False -> only alerts newer than the last triage run

ALERT_QUEUE_SQL = """
WITH flagged AS (
  SELECT a.customer_id, t.flag_reason, t.txn_id, t.txn_ts, t.amount_cents, t.channel, t.status
    FROM FINANCE.transactions t
    JOIN FINANCE.accounts a ON a.account_id = t.account_id
   WHERE t.status IN ('FLAGGED','BLOCKED') AND t.flag_reason IS NOT NULL
     AND t.txn_ts >= SYSTIMESTAMP - NUMTODSINTERVAL(:days, 'DAY')
)
SELECT f.customer_id, cu.full_name, cu.segment, cu.country, cu.risk_rating,
       f.flag_reason AS typology,
       COUNT(*) AS flagged_txns,
       SUM(f.amount_cents) AS exposure_cents,
       MIN(f.txn_ts) AS window_start, MAX(f.txn_ts) AS window_end,
       COUNT(DISTINCT f.channel) AS channels,
       SUM(CASE WHEN f.status = 'BLOCKED' THEN 1 ELSE 0 END) AS blocked_txns,
       (SELECT COUNT(*) FROM FINANCE.sar_reports s WHERE s.customer_id = f.customer_id) AS prior_sars
  FROM flagged f
  JOIN FINANCE.customers cu ON cu.customer_id = f.customer_id
 GROUP BY f.customer_id, cu.full_name, cu.segment, cu.country, cu.risk_rating, f.flag_reason
 ORDER BY cu.risk_rating DESC, SUM(f.amount_cents) DESC
"""


def alert_queue(lookback_days=ALERT_LOOKBACK_DAYS, ignore_watermark=IGNORE_WATERMARK):
    """The alerts this run should work, one row per (customer, AML typology)."""
    alerts = _rows(ALERT_QUEUE_SQL, days=lookback_days)
    if not ignore_watermark:
        with agent_conn.cursor() as cur:
            cur.execute("SELECT MAX(window_end) FROM aml_triage")
            watermark = cur.fetchone()[0]
        if watermark:
            print(f"watermark: only alerts newer than {watermark:%Y-%m-%d %H:%M}")
            alerts = [a for a in alerts if a["window_end"] > watermark]
    return alerts


_queue = alert_queue()
print(f"{len(_queue)} alert(s) in the {ALERT_LOOKBACK_DAYS}-day window — "
      f"{_money(sum(a['exposure_cents'] for a in _queue))} of flagged activity, "
      f"{'showing the top ' + str(TRIAGE_LIMIT) if len(_queue) > TRIAGE_LIMIT else 'all'}")
for a in _queue[:TRIAGE_LIMIT]:
    print(f"  [risk {a['risk_rating']:>3}] {a['full_name']:<18} {a['typology']:<18} "
          f"txns={a['flagged_txns']:>2}  exposure={_money(a['exposure_cents']):>14}  "
          f"blocked={a['blocked_txns']}  prior SARs={a['prior_sars']}")

## 12.4 The evidence pack — what the model is allowed to see

Assembled in SQL, and the *only* thing the model receives: the alert header, the flagged transactions, the surrounding activity that gives them meaning (the inbound wire before the cash-out, the merchants behind a high-risk corridor), the account history, and the prior SARs. The model does not get to choose its evidence on the happy path — it can only ask for more (`search_knowledge`, `run_sql`) inside its budget. Every decision later in this part is traceable back to lines like these.

In [ ]:
# §12.4 — Assemble one alert bundle from SQL. Show the first alert the run will work.
TYPOGRAPHY = {
    "STRUCTURING":       "$10,000 CTR threshold - deposits split just under it, repeated in a short window.",
    "GEO_VELOCITY":      "Impossible travel - one card, far-apart regions, hours apart (card cloning / takeover).",
    "HIGH_RISK_COUNTRY": "Wires to elevated-risk corridors (crypto, gold & forex, casinos, remittance) after an inbound credit.",
    "RAPID_CASH_OUT":    "Large inbound wire, then rapid ATM withdrawals draining the account within ~48 hours.",
    "LARGE_CASH_DEPOSIT": "A single cash deposit above $50,000 with no plausible source of funds.",
}


def evidence_pack(alert, max_txns=8):
    """Everything the agent is allowed to see about one alert. All of it from SQL —
    and all of it scoped to the alert's own window, so the numbers the model reasons
    over match the numbers in the header. Older flagged activity is summarised as
    history instead of being folded into this alert's totals."""
    customer_id, typology = alert["customer_id"], alert["typology"]
    lines = [f"ALERT  customer {customer_id} | {alert['full_name']} | {alert['segment']} | "
             f"{alert['country']} | risk_rating {alert['risk_rating']}/100",
             f"RULE   {typology} — {TYPOGRAPHY.get(typology, '')}",
             f"QUEUE  {alert['flagged_txns']} flagged txn(s), {_money(alert['exposure_cents'])} total, "
             f"{alert['window_start']:%Y-%m-%d} to {alert['window_end']:%Y-%m-%d}, "
             f"{alert['channels']} channel(s), {alert['blocked_txns']} blocked attempt(s)",
             "FLAGGED TRANSACTIONS IN THIS WINDOW (newest first; amounts in USD):"]
    for t in _rows("""
            SELECT t.txn_id, TO_CHAR(t.txn_ts,'YYYY-MM-DD HH24:MI') AS seen_at, t.amount_cents,
                   t.channel, t.txn_type, t.status, t.region, m.name AS merchant
              FROM FINANCE.transactions t
              JOIN FINANCE.accounts a ON a.account_id = t.account_id
              LEFT JOIN FINANCE.merchants m ON m.merchant_id = t.merchant_id
             WHERE a.customer_id = :cid AND t.status IN ('FLAGGED','BLOCKED')
               AND t.flag_reason = :typo AND t.txn_ts >= :wstart
             ORDER BY t.txn_ts DESC FETCH FIRST :n ROWS ONLY""",
                 cid=customer_id, typo=typology, wstart=alert["window_start"], n=max_txns):
        lines.append(f"  txn {t['txn_id']:>5}  {t['seen_at']}  {_money(t['amount_cents']):>13}  "
                     f"{t['channel']}/{t['txn_type']}  status={t['status']}  "
                     f"region={t['region']}  merchant={t['merchant'] or '-'}")

    history = _rows("""
            SELECT COUNT(*) AS older_txns,
                   TO_CHAR(MIN(t.txn_ts),'YYYY-MM-DD') AS first_seen,
                   TO_CHAR(MAX(t.txn_ts),'YYYY-MM-DD') AS last_seen
              FROM FINANCE.transactions t
              JOIN FINANCE.accounts a ON a.account_id = t.account_id
             WHERE a.customer_id = :cid AND t.status IN ('FLAGGED','BLOCKED')
               AND (t.flag_reason <> :typo OR t.flag_reason IS NULL OR t.txn_ts < :wstart)""",
                     cid=customer_id, typo=typology, wstart=alert["window_start"])[0]
    if history["older_txns"]:
        lines.append(f"FLAGGED HISTORY (outside this window): {history['older_txns']} earlier flagged "
                     f"txn(s), {history['first_seen']} to {history['last_seen']} — context only; the "
                     f"window above is what this decision covers.")

    lines.append("RECENT ACTIVITY, ANY STATUS (top 6 by amount, last 30 days — this is the context a "
                 "single flagged row cannot carry):")
    for t in _rows("""
            SELECT t.txn_id, TO_CHAR(t.txn_ts,'YYYY-MM-DD HH24:MI') AS seen_at, t.amount_cents,
                   t.channel, t.txn_type, t.status, m.name AS merchant
              FROM FINANCE.transactions t
              JOIN FINANCE.accounts a ON a.account_id = t.account_id
              LEFT JOIN FINANCE.merchants m ON m.merchant_id = t.merchant_id
             WHERE a.customer_id = :cid AND t.txn_ts >= SYSTIMESTAMP - INTERVAL '30' DAY
             ORDER BY t.amount_cents DESC FETCH FIRST 6 ROWS ONLY""", cid=customer_id):
        lines.append(f"  txn {t['txn_id']:>5}  {t['seen_at']}  {_money(t['amount_cents']):>13}  "
                     f"{t['channel']}/{t['txn_type']}  status={t['status']}  "
                     f"merchant={t['merchant'] or '-'}")

    for a in _rows("""
            SELECT a.account_id, a.account_type, a.currency, a.balance_cents, b.city, b.region
              FROM FINANCE.accounts a JOIN FINANCE.branches b ON b.branch_id = a.branch_id
             WHERE a.customer_id = :cid ORDER BY a.opened_ts""", cid=customer_id):
        lines.append(f"ACCOUNT #{a['account_id']} {a['account_type']} {a['currency']} "
                     f"balance {_money(a['balance_cents'])} ({a['city']}, {a['region']})")

    flow = _rows("""
            SELECT SUM(CASE WHEN t.txn_type IN ('DEPOSIT','WIRE_IN') THEN t.amount_cents ELSE 0 END) AS money_in,
                   SUM(CASE WHEN t.txn_type IN ('WITHDRAWAL','WIRE_OUT') THEN t.amount_cents ELSE 0 END) AS money_out
              FROM FINANCE.transactions t JOIN FINANCE.accounts a ON a.account_id = t.account_id
             WHERE a.customer_id = :cid AND t.txn_ts >= SYSTIMESTAMP - INTERVAL '30' DAY""",
                 cid=customer_id)[0]
    lines.append(f"30-DAY FLOW: money-in {_money(flow['money_in'])} | money-out {_money(flow['money_out'])}")

    for s in _rows("""
            SELECT sar_id, reason_code, status, TO_CHAR(filed_ts,'YYYY-MM-DD') AS filed,
                   SUBSTR(narrative, 1, 180) AS narrative
              FROM FINANCE.sar_reports WHERE customer_id = :cid ORDER BY filed_ts DESC""",
                   cid=customer_id):
        lines.append(f"PRIOR SAR {s['sar_id']}: {s['reason_code']} status={s['status']} "
                     f"filed {s['filed']} — {s['narrative']}")
    return "\n".join(lines)


_sample_pack = evidence_pack(_queue[0])
print(_sample_pack)
print(f"\n[{len(_sample_pack.splitlines())} lines, {len(_sample_pack)} chars — "
      f"the model's entire view of this customer]")

## 12.5 The decision policy — three verbs and a JSON contract

| Decision | What it authorises | What drives it |
|---|---|---|
| `ESCALATE` | recommend a SAR (new, or an update to an open one) | repetition, blocked attempts, prior SAR, threshold proximity |
| `KYC_REVIEW` | investigate before filing — source-of-funds request, KYC refresh | a real signal on thin evidence |
| `DISMISS` | close the alert **with a written reason** an examiner would accept | activity explained by legitimate behaviour |

The model returns JSON; the harness validates it. Two rules matter:

1. A decision outside the vocabulary becomes **`REVIEW_REQUIRED`** — a human works it. Not an exception, not a silent pass-through.
2. An unknown `sar_reason_code` falls back to the alert's own typology. *The model owns the reasoning; the harness owns the enum.*

In [ ]:
# §12.5 — The policy the model works under, and the validator that enforces it.
TRIAGE_SYSTEM_PROMPT = """You are the Meridian Bank AML triage analyst operating inside an agent harness.
You receive ONE alert. Decide what the bank should do with it.

Decide exactly one:
  ESCALATE   — the evidence meets the filing bar: recommend a SAR (new, or an update to an open one).
  KYC_REVIEW — a real signal, but not enough to file: request source-of-funds / refresh KYC.
  DISMISS    — the activity is explained by legitimate behaviour; write down why an examiner would agree.

Rules:
- Consult your memory first: call search_knowledge for this customer's prior case decisions and the schema
  notes before you decide, and call run_sql if the evidence leaves a question the SQL can answer.
- Judge only the evidence provided; never invent customers, accounts, or amounts.
- Quantify: cite transaction ids and USD amounts (amounts in the alert are already in dollars).
- Weigh repetition, blocked attempts, proximity to reporting thresholds ($10,000 CTR / $50,000),
  prior SARs, and risk_rating.
- If the evidence is thin, say so and choose KYC_REVIEW.

Reply with ONE JSON object and nothing else:
{"decision":"ESCALATE|KYC_REVIEW|DISMISS","confidence":0.0-1.0,"rationale":"<=60 words, cite txn ids and $ amounts",
 "indicators":["..."],"recommended_next_action":"one sentence",
 "sar_reason_code":"STRUCTURING|GEO_VELOCITY|HIGH_RISK_COUNTRY|RAPID_CASH_OUT|LARGE_CASH_DEPOSIT|null"}"""

DECISIONS = ("ESCALATE", "KYC_REVIEW", "DISMISS")


def _extract_json(text):
    """Models wrap JSON in prose often enough that the harness must not trust `json.loads`."""
    match = re.search(r"\{.*\}", text or "", re.S)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        try:
            return json.loads(re.sub(r"\s+", " ", match.group(0)))
        except json.JSONDecodeError:
            return None


def _validate_decision(payload, alert):
    """Clamp whatever the model said to something the bank allows. Never raises."""
    if not isinstance(payload, dict):
        return {"decision": "REVIEW_REQUIRED", "confidence": 0.0,
                "rationale": "Model reply was not valid JSON; a human must work this alert.",
                "indicators": [], "recommended_next_action": "Human review of the raw reply.",
                "sar_reason_code": alert["typology"]}
    decision = str(payload.get("decision", "")).strip().upper()
    if decision not in DECISIONS:
        decision = "REVIEW_REQUIRED"
    try:
        confidence = max(0.0, min(1.0, float(payload.get("confidence", 0.0))))
    except (TypeError, ValueError):
        confidence = 0.0
    reason = str(payload.get("sar_reason_code") or "").strip().upper()
    if reason not in TYPOGRAPHY:
        reason = alert["typology"]           # the harness owns the enum
    return {
        "decision": decision,
        "confidence": confidence,
        "rationale": str(payload.get("rationale") or "")[:900],
        "indicators": [str(i)[:60] for i in (payload.get("indicators") or [])][:6],
        "recommended_next_action": str(payload.get("recommended_next_action") or "")[:380],
        "sar_reason_code": reason,
    }


for _probe, _expected in [({"decision": "escalate", "confidence": 1.7, "sar_reason_code": "made_up"}, "ESCALATE"),
                          ("not json at all", "REVIEW_REQUIRED")]:
    _got = _validate_decision(_probe, {"typology": "STRUCTURING"})["decision"]
    assert _got == _expected, f"validator drift: {_got} != {_expected}"
print("✅ decision validator holds: unknown decisions become REVIEW_REQUIRED, confidence is clamped, "
      "reason codes are restricted to the bank's typologies")

## 12.6 `triage_alert` — the decision, recorded

One alert in, one ledger row and one memory out. The function reuses **`agent_turn`** — the loop from TODO 9 — unchanged: same context assembly, same tool dispatch, same budgets. **Autonomy is not a different loop**; it is the same loop with a trigger and a record.

Three details are the difference between a demo and something a bank could run: the triage instruction travels in the **user message** (so the loop stays reusable, and the thread remembers the alerts it already worked); every decision is written **twice on purpose** (`AML_TRIAGE` for the examiner, a `case_decision` memory for the agent's next run); and model calls are **metered** so §12.8 can report what autonomy cost.

In [ ]:
# §12.6 — Meter the model, then work one alert end to end.
_USAGE = {"calls": 0, "tokens": 0, "seconds": 0.0}

if not globals().get("_CHAT_METERED"):
    _unmetered_chat = chat

    def chat(*args, **kwargs):                     # noqa: F811 — metered wrapper around the Part 1 client
        started = time.time()
        response = _unmetered_chat(*args, **kwargs)
        _USAGE["calls"] += 1
        _USAGE["seconds"] += time.time() - started
        _USAGE["tokens"] += getattr(getattr(response, "usage", None), "total_tokens", 0) or 0
        return response

    _CHAT_METERED = True
print("model-call meter installed — every LLM round-trip from here on is counted")

TRIAGE_THREAD = "aml-triage-desk"


def triage_alert(alert, thread_id=TRIAGE_THREAD, verbose=True):
    """Work one alert: the agent decides, the harness validates and records."""
    prompt = (TRIAGE_SYSTEM_PROMPT + "\n\n--- ALERT ---\n" + evidence_pack(alert)
              + "\n--- END ALERT ---\nDecide now. Reply with the JSON object only.")
    reply = agent_turn(prompt, thread_id=thread_id, max_iterations=6, budget_seconds=120.0, verbose=verbose)
    decision = _validate_decision(_extract_json(reply), alert)

    with agent_conn.cursor() as cur:
        cur.execute("""
            MERGE INTO aml_triage t
            USING (SELECT :cid AS customer_id, :typo AS typology FROM dual) s
               ON (t.customer_id = s.customer_id AND t.typology = s.typology)
            WHEN MATCHED THEN UPDATE SET
                 customer_name = :name, risk_rating = :risk, flagged_txns = :n,
                 exposure_cents = :exp, window_start = :ws, window_end = :we,
                 decision = :dec, confidence = :conf, rationale = :rat,
                 next_action = :nxt, sar_reason = :sar,
                 decided_at = SYSTIMESTAMP, decided_by = :actor
            WHEN NOT MATCHED THEN INSERT
                 (customer_id, typology, customer_name, risk_rating, flagged_txns, exposure_cents,
                  window_start, window_end, decision, confidence, rationale, next_action, sar_reason,
                  decided_at, decided_by)
                 VALUES (:cid, :typo, :name, :risk, :n, :exp, :ws, :we, :dec, :conf, :rat, :nxt, :sar,
                         SYSTIMESTAMP, :actor)""",
            cid=alert["customer_id"], typo=alert["typology"], name=alert["full_name"],
            risk=alert["risk_rating"], n=alert["flagged_txns"], exp=alert["exposure_cents"],
            ws=alert["window_start"], we=alert["window_end"], dec=decision["decision"],
            conf=decision["confidence"], rat=decision["rationale"],
            nxt=decision["recommended_next_action"], sar=decision["sar_reason_code"],
            actor="aml-triage-harness")
    agent_conn.commit()

    write_facts([Fact(
        kind="case_decision",
        subject=f"case:{alert['customer_id']}:{alert['typology']}",
        body=(f"AML triage {decision['decision']} for customer {alert['customer_id']} "
              f"({alert['full_name']}, risk {alert['risk_rating']}/100), typology {alert['typology']}, "
              f"exposure {_money(alert['exposure_cents'])}: {decision['rationale']} "
              f"Next action: {decision['recommended_next_action']}"),
        metadata={"source": "aml_triage", "decision": decision["decision"],
                  "typology": alert["typology"], "customer_id": int(alert["customer_id"]),
                  "confidence": decision["confidence"]},
    )])
    return {**alert, **decision}


print("triage_alert ready — agent_turn decides, the ledger and OAMP memory record")

## 12.7 Run it — the bank's morning queue

Three alerts by default, each a full model round-trip inside the bounded loop. The run prints one line per alert — decision, confidence, reason — then dumps the ledger as an examiner's SQL client would read it. A failure inside one alert is recorded and the run moves on: the difference between "the desk ran" and "the desk stopped because one customer's data was odd".

In [ ]:
# §12.7 — Work the queue and print the ledger.
def run_triage(alerts=None, limit=TRIAGE_LIMIT, thread_id=TRIAGE_THREAD):
    queue = list(alerts if alerts is not None else alert_queue())[:limit]
    if not queue:
        print("Queue empty — no alerts in the lookback window "
              "(set IGNORE_WATERMARK = True in §12.3 to re-triage the same window).")
        return []
    records, failures = [], []
    started = time.time()
    for i, alert in enumerate(queue, 1):
        print(f"\n[{i}/{len(queue)}] customer {alert['customer_id']} · {alert['typology']} · "
              f"{_money(alert['exposure_cents'])} flagged exposure")
        try:
            record = triage_alert(alert, thread_id=thread_id)
        except Exception as exc:                    # one bad alert must not end the desk's morning
            failures.append((alert, exc))
            print(f"   ! triage failed: {type(exc).__name__}: {str(exc)[:200]}")
            continue
        records.append(record)
        print(f"   -> {record['decision']:<16} confidence {record['confidence']:.2f}")
        print(f"      {record['rationale']}")
        print(f"      next: {record['recommended_next_action']}")
    print(f"\n{len(records)} decision(s) in {time.time() - started:.0f}s · "
          f"{_USAGE['calls']} model call(s) · {_USAGE['tokens']:,} tokens")
    if failures and not records:
        print("\nEvery alert failed. If the traceback says a TODO is still a stub, finish the TODOs "
              "above before running Part 12.")
    return records


_records = run_triage()

print("\nLedger — AGENT.AML_TRIAGE (what a reviewer or examiner would query):")
for row in _rows("""
        SELECT customer_id, customer_name, typology, decision, confidence,
               exposure_cents, next_action
          FROM aml_triage ORDER BY decided_at DESC"""):
    print(f"  cust {row['customer_id']:>3}  {row['customer_name']:<18} {row['typology']:<18} "
          f"{row['decision']:<16} conf {row['confidence']:.2f}  {_money(row['exposure_cents']):>14}  "
          f"{(row['next_action'] or '')[:60]}")

## 12.8 The impact board — what this is worth to Meridian Bank

An AML programme is measured on a short list: alerts worked, time-to-decision, filings made, evidence that survives an examination — and what it costs in people to produce all three. The board computes those numbers from the ledger and `FINANCE`.

One input cannot come from the database, so it is a labelled knob: `MANUAL_MINUTES_PER_ALERT` (45 by default — change it and re-run). Everything above that row is computed, and the last block states plainly what the harness did **not** do.

In [ ]:
# §12.8 — The morning briefing, computed from the ledger + FINANCE.
MANUAL_MINUTES_PER_ALERT = 45.0   # ASSUMPTION, not a measurement — change it and re-run.

_by_decision = _rows("""
        SELECT decision, COUNT(*) AS alerts, SUM(exposure_cents) AS exposure_cents
          FROM aml_triage GROUP BY decision ORDER BY alerts DESC, exposure_cents DESC""")
_total_alerts = sum(r["alerts"] for r in _by_decision)
_decision_of = {r["decision"]: r for r in _by_decision}
_escalated = _decision_of.get("ESCALATE", {"alerts": 0, "exposure_cents": 0})
_review = _decision_of.get("REVIEW_REQUIRED", {"alerts": 0, "exposure_cents": 0})
_confidence = _rows("SELECT ROUND(AVG(confidence), 2) AS avg_conf FROM aml_triage "
                    "WHERE decision <> 'REVIEW_REQUIRED'")[0]["avg_conf"]
_sars = _rows("SELECT COUNT(*) AS total, SUM(CASE WHEN status = 'FILED' THEN 1 ELSE 0 END) AS filed "
              "FROM finance.sar_reports")[0]
_ages = _rows("SELECT ROUND(MAX(CAST(SYSTIMESTAMP AS DATE) - CAST(window_end AS DATE))) AS oldest, "
              "       ROUND(AVG(CAST(SYSTIMESTAMP AS DATE) - CAST(window_end AS DATE))) AS average "
              "  FROM aml_triage")[0]
_age_line = (f"{int(_ages['oldest'])}d  (avg {int(_ages['average'])}d)"
             if _ages["oldest"] is not None else "n/a — the ledger is empty; run §12.7 first")
_manual_hours = _total_alerts * MANUAL_MINUTES_PER_ALERT / 60.0

print("MERIDIAN BANK · AML ALERT TRIAGE · MORNING BRIEFING")
print("=" * 72)
print(f"  alerts decided by the harness .......... {_total_alerts}")
for row in _by_decision:
    bar = "█" * max(1, int(32 * row["alerts"] / max(_total_alerts, 1)))
    print(f"    {row['decision']:<16} {row['alerts']:>3}  {bar}  {_money(row['exposure_cents'])}")
print(f"  flagged exposure escalated to compliance {_money(_escalated['exposure_cents'])}")
print(f"  average confidence on decided alerts ... {_confidence if _confidence is not None else 'n/a'}")
print(f"  oldest alert worked (now - newest txn) . {_age_line}")
print("-" * 72)
print(f"  harness cost ........................... {_USAGE['calls']} model call(s) · "
      f"{_USAGE['tokens']:,} tokens · {_USAGE['seconds']:.0f}s of model wall-clock")
print(f"  equivalent manual effort ............... {_manual_hours:.1f}h at "
      f"{MANUAL_MINUTES_PER_ALERT:.0f} min/alert (assumption)")
print(f"  SAR history on file .................... {_sars['total']} report(s), {_sars['filed']} filed")
if _review["alerts"]:
    print(f"  human review required .................. {_review['alerts']} alert(s) — "
          f"the model's reply did not validate")
print("=" * 72)
print("What the harness did NOT do: file anything, write to FINANCE, or see data its persona "
      "is not entitled to.")
print("The recommendation is a decision record; a human signs the filing. That boundary is the "
      "reason this can run every morning.")

## 12.9 Close the loop — the agent remembers its own decisions

The decisions are memories (`kind="case_decision"`), not just rows, so the same `agent_turn` can answer questions about its own morning — which is the difference between a decision log and a colleague who was there. Watch `search_knowledge` fire: the answer is grounded in the rationales it wrote minutes ago, not re-derived from SQL.

In [ ]:
# §12.9 — Ask the agent about its own triage run. No new SQL: this is memory recall.
_question = ("You triaged part of Meridian Bank's AML alert queue this morning. From your own case "
             "decisions: which customers did you escalate, what is the total exposure now under review, "
             "and which single case should a human reviewer pick up first — and why?")
print("USER:", _question)
print("\nAGENT:", agent_turn(_question, thread_id=TRIAGE_THREAD, max_iterations=6, budget_seconds=120.0))

## 12.10 From one morning to a running programme

| Next step | How, in this stack |
|---|---|
| **Run it every morning** | a `DBMS_SCHEDULER` job, like the app's scan scheduler (`app/backend/db/scheduler_setup.py`) |
| **Make it incremental** | `IGNORE_WATERMARK = False` in §12.3 — the second run of the day reports **0 new alerts** |
| **Keep a human in the loop** | `REVIEW_REQUIRED` already routes unvalidated output to a person; sample a percentage of `DISMISS` decisions |
| **Watch precision, not volume** | `SELECT decision, COUNT(*) FROM aml_triage GROUP BY decision` is the false-positive rate over time |
| **Prove it to an examiner** | every row carries its evidence window, rationale, and confidence — replayable from OAMP |
| **Scope it by identity** | run the desk as `compliance.officer` vs `analyst.east` and the same queue returns different rows ([Part 8](docs/part-8-deep-data-security.md)) |

The honest limits: a model that decides can be wrong at scale, the policy is only as good as its thresholds, and the seeded queue is true-positive-weighted. The claim is not that the agent is right — it is that its judgement arrives with **evidence, a reason, a record, and a budget**.

**See it as a product:** [http://localhost:3000](http://localhost:3000) runs on the same store this notebook just wrote to.

<!-- workshop:part-12:end -->

# Closing thoughts — and the running app

You built the harness:

- **Long-term memory** — OAMP threads, memories, and context cards, filled by a scanner that mines Oracle's catalog.
- **Retrieval** — in-database embeddings, cosine search, a cross-encoder reranker, and hybrid Oracle Text + vector RRF.
- **Vector-indexed `toolbox` and `skillbox`** — the right tool or playbook on demand, not the whole registry.
- **Context engineering** — a compact prompt assembled from skills, memories, retrieved knowledge, and the question.
- **The agent loop** — the bounded call → dispatch → update cycle.

## See it running

The Codespace started a **Flask + React deployment of this same harness**: **http://localhost:3000** (check the **PORTS** tab if no preview opened). Same Oracle, same memory store, same `toolbox` and `skillbox` you just populated. It adds a chat UI with a bubble per tool call, a live memory pane, a persona selector that changes which rows the kernel returns, Oracle MLE compute, DBFS, duality views, spatial search, and scheduled automations. Architecture: [`app/README.md`](app/README.md).

## Deploy it elsewhere

To run this harness against an Oracle that is not the workshop Codespace, run the same provisioning this one did: `app/scripts/bootstrap.py` → `seed.py` → `setup_advanced.py` → `setup_deep_security.py`, all idempotent ([`docs/part-1-setup.md`](docs/part-1-setup.md)).

---

# The capstone — decisions, not just answers

**Part 12** turned the harness outward: it worked Meridian Bank's AML alert queue end to end, decided with a reason and a confidence, wrote every decision to `AGENT.AML_TRIAGE` and to memory, and reported what the run was worth to the bank.

It is the same loop you built in TODO 9. A **trigger**, a **record**, and a **budget** are what turn a chat window into something a compliance desk could run every morning — and the split between the model's judgement and the harness's state is what makes it auditable.